### Preamble

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [ ]:
!pip install -q transformers peft evaluate tomli -U accelerate -U bitsandbytes -q rank-bm25


In [ ]:
!pip install --upgrade torch torchvision torchaudio

In [ ]:
import os
import re
import json
import time
import random
import datetime
import numpy as np
import pandas as pd
from tqdm import tqdm
from sklearn.model_selection import train_test_split

import torch
from torch.utils.data import Dataset, DataLoader
from torch.optim import AdamW
from transformers import (
    AutoTokenizer,
    AutoModel,
    get_cosine_schedule_with_warmup,
)

from sklearn.metrics import accuracy_score
import evaluate

from peft import LoraConfig, get_peft_model


ModuleNotFoundError: No module named 'evaluate'

In [ ]:
torch.set_default_dtype(torch.float32)

In [ ]:
from huggingface_hub import login
HF_ACCESS_TOKEN = "Add your token here"
login(HF_ACCESS_TOKEN)

### Language

In [ ]:
language = "English" # "English" # "Arabic"

In [ ]:
train = f"/content/drive/MyDrive/CheckThat Task2/{language}/train.jsonl"

if language == "English":
  test = f"/content/drive/MyDrive/CheckThat Task2/{language}/clef_2026_final_english_test.json"
elif language == "Spanish":
  test = f"/content/drive/MyDrive/CheckThat Task2/{language}/clef_spanish_test_final.json"
elif language == "Arabic":
  test = f"/content/drive/MyDrive/CheckThat Task2/{language}/clef_2026_final_arabic_test.json"

### Data

In [ ]:
import pandas as pd
import json
import random
import re


unknown_counter = 0
UNKNOWN_LIMIT = 150

def remove_label_pattern(text):
    justification = re.sub(r"(\[?\s*Justification\s*\]?:?\s*)|(\[Label\]:\s*(True|False|Conflicting))", "", text, flags=re.IGNORECASE).strip()
    return justification.replace("\n", " ")

def sample_training_examples(row):
    global unknown_counter

    label = row["label"].lower()
    verdict_list = [v.lower() for v in row["Verdict_list"]]

    # Step 1: Identify indices of different types
    correct_indices = [i for i, v in enumerate(verdict_list) if v == label]
    true_indices = [i for i, v in enumerate(verdict_list) if v == "true" and v != label]
    false_indices = [i for i, v in enumerate(verdict_list) if v == "false" and v != label]
    conflicting_indices = [i for i, v in enumerate(verdict_list) if v == "conflicting" and v != label]
    unknown_indices = [i for i, v in enumerate(verdict_list) if v == "unknown"]

    selected_indices = []

    # Step 2: Sample correct ones if available
    if len(correct_indices) >= 2:
        selected_indices.extend(random.sample(correct_indices, 2))
        num_remaining = 4
    elif len(correct_indices) == 1:
        selected_indices.append(correct_indices[0])
        num_remaining = 5
    else:
        num_remaining = 6

    # Step 3: Fill remaining slots with diverse wrong answers
    wrong_indices = []
    if label != "true" and true_indices:
        wrong_indices.append(random.choice(true_indices))
    if label != "false" and false_indices:
        wrong_indices.append(random.choice(false_indices))
    if label != "conflicting" and conflicting_indices:
        wrong_indices.append(random.choice(conflicting_indices))

    # Step 4: Ensure diversity while filling remaining slots
    wrong_indices = list(set(wrong_indices))  # Remove duplicates
    needed = num_remaining - len(wrong_indices)

    # Add extra wrong ones if needed
    all_wrong_indices = true_indices + false_indices + conflicting_indices
    random.shuffle(all_wrong_indices)
    wrong_indices.extend(all_wrong_indices[:needed])

    selected_indices.extend(wrong_indices[:num_remaining])

    # Step 5: Handle case where all values are "unknown"
    if not selected_indices and unknown_indices:
        # If everything is unknown, just take 5 unknowns
        selected_indices = random.sample(unknown_indices, min(5, len(unknown_indices)))
    elif unknown_indices and unknown_counter < UNKNOWN_LIMIT:
        # Only pop if there is something to pop
        if selected_indices:
            selected_indices.pop()
        selected_indices.append(random.choice(unknown_indices))
        unknown_counter += 1

    return selected_indices


# data = pd.read_json("reasoning_traces/quantemp_English_train.jsonl", lines=True)
data = pd.DataFrame(pd.read_json(f"/content/drive/MyDrive/CheckThat Task2/{language}/spanish_train.json"))

# Apply function to each row. sampled_indices contain the indices of the sampled dataset
data["sampled_indices"] = data.apply(sample_training_examples, axis=1)


final_training_data = []

for idx in range(len(data)):
    class_label = 0
    item = data.loc[idx]

    label = item['label'].lower()



    for decoding_idx, decoding_sample in enumerate(item["sampled_indices"]):

        justification = remove_label_pattern(item["Reasoning_traces"][decoding_sample])
        verdict = item['Verdict_list'][decoding_sample].lower()

        sample_id = str(idx) + "_" + chr(97 + decoding_idx)


        if verdict == label:
            class_label = 1
        else:
            class_label = 0



        # Handles empty justification
        if len(justification.split(" ")) < 3:
            continue


        final_training_data.append({
            "sample_id": sample_id,
            "input_text": f"Claim: {item['claim']}\nVerdict: {verdict}\nJustification: {justification}",
            "Label": label,
            "Verdict": verdict,
            "Class": class_label})

print(len(final_training_data))
# with open("output/training_data_for_RM/English_train.json", "w") as fp:
#    json.dump(final_training_data, fp, indent = 4)
pd.DataFrame(final_training_data).to_json(
     f"/content/drive/MyDrive/CheckThat Task2/{language}/train.jsonl",
     orient="records",
     lines=True,
     force_ascii=False
 )

print("Finshed preprocessing the data.")

9038
Finshed preprocessing the data.


In [ ]:
import pandas as pd
import json

# Load your raw nested JSON
file_path = f"/content/drive/MyDrive/CheckThat Task2/{language}/clef_2026_checkthat_english_train.json"
with open(file_path, "r") as f:
    data = json.load(f)

df = pd.DataFrame(data)

# 1. Ground Truth Distribution
print("=== Ground Truth Label Distribution ===")
print(df['label'].str.lower().value_counts(normalize=True) * 100)

# 2. Generator Output Distribution
all_generated_verdicts = []
for verdicts in df['Verdict_list']:
    all_generated_verdicts.extend([v.lower() for v in verdicts])

print("\n=== Generated Traces Verdict Distribution ===")
print(pd.Series(all_generated_verdicts).value_counts(normalize=True) * 100)

# 3. The "Trap" Analysis (How often are there ZERO correct traces?)
zero_correct_count = 0
for idx, row in df.iterrows():
    ground_truth = row['label'].lower()
    verdicts = [v.lower() for v in row['Verdict_list']]
    if ground_truth not in verdicts:
        zero_correct_count += 1

print(f"\n=== Impossible Cases ===")
print(f"Claims where the generator produced ZERO correct traces: {zero_correct_count} out of {len(df)}")
print(f"Percentage of dataset permanently trapped: {(zero_correct_count/len(df))*100:.2f}%")

=== Ground Truth Label Distribution ===
label
false          58.078125
conflicting    23.562500
true           18.359375
Name: proportion, dtype: float64

=== Generated Traces Verdict Distribution ===
false          39.850417
conflicting    31.103763
true           29.045820
Name: proportion, dtype: float64

=== Impossible Cases ===
Claims where the generator produced ZERO correct traces: 1917 out of 6400
Percentage of dataset permanently trapped: 29.95%


### Preprocessing

In [ ]:
f1_metric = evaluate.load("f1")

def format_time(elapsed):
    elapsed_rounded = int(round(elapsed))
    return str(datetime.timedelta(seconds=elapsed_rounded))


def remove_label_pattern(text):
    text = re.sub(
        r"(\[?\s*Justification\s*\]?:?\s*)|(\[Label\]:\s*(True|False|Conflicting))",
        "",
        text,
        flags=re.IGNORECASE,
    ).strip()
    return text.replace("\n", " ")


def print_trainable_parameters(model):
    trainable_params, all_param = 0, 0
    for _, param in model.named_parameters():
        all_param += param.numel()
        if param.requires_grad:
            trainable_params += param.numel()

    print(
        f"trainable params: {trainable_params} || "
        f"all params: {all_param} || "
        f"trainable%: {100 * trainable_params / all_param:.2f}"
    )


In [ ]:
class CustomClassifier(torch.nn.Module):
    def __init__(
        self,
        model_name,
        num_labels=1,
        hidden_dim=None,
        dropout_value=0.1,
        freeze_base_layer=True,
        use_lora=False,
        is_base_encoder=True,
        lora_rank=8,
        lora_alpha=16,
        quantization_config=None,
        device_map="auto",
        token=None
    ):
        super().__init__()

        self.model = AutoModel.from_pretrained(
            model_name,
            quantization_config=quantization_config,
            device_map=device_map,
            token=token
        )
        self.is_base_encoder = is_base_encoder

        if freeze_base_layer:
            for param in self.model.parameters():
                param.requires_grad = False

        if use_lora:
            lora_config = LoraConfig(
                r=lora_rank,
                lora_alpha=lora_alpha,
                target_modules=["q_proj", "k_proj", "v_proj"],
                lora_dropout=0.0,
                bias="none",
            )
            self.model = get_peft_model(self.model, lora_config)

        hidden_size = self.model.config.hidden_size

        if hidden_dim:
            self.classifier = torch.nn.Sequential(
                torch.nn.Linear(hidden_size, hidden_dim),
                torch.nn.ReLU(),
                torch.nn.Dropout(dropout_value),
                torch.nn.Linear(hidden_dim, num_labels),
            )
        else:
            self.classifier = torch.nn.Linear(hidden_size, num_labels)

    def forward(self, input_ids, attention_mask):
        outputs = self.model(input_ids=input_ids, attention_mask=attention_mask)

        if self.is_base_encoder:
            pooled_output = outputs.pooler_output
        else:
            pooled_output = outputs.last_hidden_state[:, -1, :]
        pooled_output = pooled_output.to(torch.device("cuda"), dtype=torch.float)
        #self.classifier = self.classifier.to(torch.device("cuda"),torch.float16)
        logits = self.classifier(pooled_output)
        return logits

In [ ]:
class TextDataset(Dataset):
    def __init__(self, dataframe, tokenizer, max_length):
        self.texts = dataframe["input_text"].tolist()
        self.labels = dataframe["Class"].tolist()
        self.tokenizer = tokenizer
        self.tokenizer.pad_token = self.tokenizer.eos_token
        self.max_length = max_length

    def __len__(self):
        return len(self.texts)

    def __getitem__(self, idx):
        encoding = self.tokenizer(
            self.texts[idx],
            truncation=True,
            padding="max_length",
            max_length=self.max_length,
            return_tensors="pt",
        )
        item = {k: v.squeeze(0) for k, v in encoding.items()}
        item["labels"] = torch.tensor(self.labels[idx], dtype=torch.float)
        return item


In [ ]:
class TrainerModule:
    def __init__(
        self,
        model,
        train_loader,
        val_loader,
        tokenizer,
        epochs,
        lr,
        patience,
        output_dir,
    ):
        self.device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
        self.model = model.to(self.device)

        self.train_loader = train_loader
        self.val_loader = val_loader
        self.tokenizer = tokenizer
        self.epochs = epochs

        self.optimizer = AdamW(model.parameters(), lr=lr, eps=1e-8)
        self.loss_fn = torch.nn.BCEWithLogitsLoss()

        total_steps = len(train_loader) * epochs
        warmup_steps = int(0.05 * total_steps)

        self.scheduler = get_cosine_schedule_with_warmup(
            self.optimizer,
            warmup_steps,
            total_steps,
        )

        self.output_dir = output_dir
        os.makedirs(output_dir, exist_ok=True)

    def train(self):
        for epoch in range(self.epochs):
            print(f"\nEpoch {epoch+1}/{self.epochs}")
            self.model.train()

            total_loss, total_acc = 0, 0

            for batch in tqdm(self.train_loader):
                self.optimizer.zero_grad()

                input_ids = batch["input_ids"].to(self.device)
                attention_mask = batch["attention_mask"].to(self.device)
                labels = batch["labels"].to(self.device)
                logits = self.model(input_ids, attention_mask)

                loss = self.loss_fn(logits.squeeze(1), labels)

                loss.backward()
                self.optimizer.step()
                self.scheduler.step()

                total_loss += loss.item()

                preds = (torch.sigmoid(logits) >= 0.5).cpu().numpy()
                total_acc += accuracy_score(labels.cpu().numpy(), preds)

            print(f"Train Loss: {total_loss / len(self.train_loader):.4f}")
            print(f"Train Acc: {total_acc / len(self.train_loader):.4f}")

            self.evaluate(epoch)

    def evaluate(self, epoch):
        self.model.eval()
        total_loss, total_acc = 0, 0

        with torch.no_grad():
            for batch in self.val_loader:
                input_ids = batch["input_ids"].to(self.device)
                attention_mask = batch["attention_mask"].to(self.device)
                labels = batch["labels"].to(self.device)

                logits = self.model(input_ids, attention_mask)
                loss = self.loss_fn(logits.squeeze(1), labels)

                total_loss += loss.item()
                preds = (torch.sigmoid(logits) >= 0.5).cpu().numpy()
                total_acc += accuracy_score(labels.cpu().numpy(), preds)

        print(f"Val Loss: {total_loss / len(self.val_loader):.4f}")
        print(f"Val Acc: {total_acc / len(self.val_loader):.4f}")
        self.tokenizer.save_pretrained(self.output_dir)
        torch.save(
            self.model.state_dict(),
            os.path.join(self.output_dir, f"model"),
        )


In [ ]:
class VerifierEvaluator:
    def __init__(
        self,
        model_path,
        tokenizer_path,
        base_model,
        use_decomp,
        device="cuda",
        quantization_config=None,
        token=None
    ):
        super().__init__()
        self.device = torch.device(device if torch.cuda.is_available() else "cpu")
        self.use_decomp = use_decomp

        # Pass token to AutoTokenizer.from_pretrained
        self.tokenizer = AutoTokenizer.from_pretrained(tokenizer_path, token=token)
        self.tokenizer.pad_token = self.tokenizer.eos_token

        self.model = CustomClassifier(
            base_model,
            use_lora=True,
            is_base_encoder=False,
            quantization_config=quantization_config,
            token=token
        )
        self.model.load_state_dict(torch.load(model_path, map_location=self.device), strict=False)
        self.model.to(self.device)
        self.model.eval()

    def encode_input(self, claim, questions, verdict, justification, max_length=150):

        text = f"Claim: {claim}\nVerdict: {verdict}\nJustification: {justification}"

        encoding = self.tokenizer(
            text,
            truncation=True,
            padding="max_length",
            max_length=max_length,
            return_tensors="pt",
        )

        return (
            encoding["input_ids"].to(self.device),
            encoding["attention_mask"].to(self.device),
        )

    def score(self, claim, questions, verdict, justification):
        ids, mask = self.encode_input(claim, questions, verdict, justification)
        with torch.no_grad():
            return float(self.model(ids, mask).item())

# LoRA Fine-Tuning LLaMA-3.2-Instruct 1B
This notebook fine-tunes a HuggingFace LLaMA-3B-Instruct model using LoRA (PEFT).

### Training

In [ ]:
# ===== PATH =====
TRAIN_CSV = train
VAL_CSV   = test
# TEST_JSON = "/kaggle/input/YOUR_DATA/test.jsonl"

OUTPUT_DIR = f"/content/drive/MyDrive/CheckThat Task2/{language}"

BASE_MODEL = "meta-llama/Llama-3.2-1B"

MAX_LENGTH = 256
BATCH_SIZE = 4
EPOCHS = 3
LR = 1e-4


In [ ]:

train_df = pd.read_json(TRAIN_CSV, lines = True)

train_df, val_df = train_test_split(
    train_df,
    test_size=0.2,
    stratify=train_df["Class"],
    random_state=42
)
# val_df = pd.read_json(VAL_CSV, lines = True)

tokenizer = AutoTokenizer.from_pretrained(BASE_MODEL, token=HF_ACCESS_TOKEN)

train_dataset = TextDataset(train_df, tokenizer, MAX_LENGTH)
val_dataset = TextDataset(val_df, tokenizer, MAX_LENGTH)

train_loader = DataLoader(train_dataset, batch_size=BATCH_SIZE, shuffle=True)
val_loader = DataLoader(val_dataset, batch_size=BATCH_SIZE)

model = CustomClassifier(
    BASE_MODEL,
    use_lora=True,
    is_base_encoder=False,
    device_map="auto",
    token=HF_ACCESS_TOKEN
)

print_trainable_parameters(model)

trainer = TrainerModule(
    model=model,
    train_loader=train_loader,
    val_loader=val_loader,
    tokenizer = tokenizer,
    epochs=EPOCHS,
    lr=LR,
    patience=2,
    output_dir=OUTPUT_DIR,
)

trainer.train()


Loading weights:   0%|          | 0/146 [00:00<?, ?it/s]

trainable params: 1181697 || all params: 1236996097 || trainable%: 0.10

Epoch 1/3


100%|██████████| 1808/1808 [02:48<00:00, 10.73it/s]


Train Loss: 0.3564
Train Acc: 0.8473
Val Loss: 0.3074
Val Acc: 0.8844

Epoch 2/3


100%|██████████| 1808/1808 [02:47<00:00, 10.79it/s]


Train Loss: 0.2118
Train Acc: 0.9186
Val Loss: 0.2256
Val Acc: 0.9076

Epoch 3/3


100%|██████████| 1808/1808 [02:46<00:00, 10.84it/s]


Train Loss: 0.1206
Train Acc: 0.9490
Val Loss: 0.1930
Val Acc: 0.9231


### Predictions

In [ ]:
with open(VAL_CSV, "r") as f:
  test_data = json.load(f)

predictions = []

evaluator = VerifierEvaluator(
    model_path=f"{OUTPUT_DIR}/model",
    tokenizer_path=BASE_MODEL,
    base_model=BASE_MODEL,
    use_decomp=True,
)

# Example scoring
for idx, sample in enumerate(test_data):
  verdict_list = []
  verifier_score_list = []
  justification_list = []
  approved_majority_list = []
  for decoding_sample in range(1, len(sample["Reasoning_traces"]) + 1):
    justification = (
        remove_label_pattern(
            sample["Reasoning_traces"][decoding_sample - 1]
        ).split("Label:")[0]
    )
    score = evaluator.score(
    sample["claim"],
    sample.get("Questions", ""),
    sample["Verdict_list"][decoding_sample - 1].lower(),
    justification,
)
    verdict_list.append(sample["Verdict_list"][decoding_sample - 1])
    justification_list.append(justification)
    verifier_score_list.append(score)
    best_verdict = verdict_list[np.argmax(np.array(verifier_score_list))]
    predictions.append({
        "query_id": idx,
        "Claim": sample["claim"],
        "Label": sample["label"],
        "Verdict_BoN": best_verdict,
        "BoN_Verdict_list": verdict_list,
        "Reasoning_traces": justification_list,
        "score_list": verifier_score_list,
    })




Loading weights:   0%|          | 0/146 [00:00<?, ?it/s]

In [ ]:
with open(f"/content/drive/MyDrive/CheckThat Task2/{language}/clef_predictions_1B.json", "w") as fp:
  json.dump(predictions, fp, indent=4)

# Pairwise Ranking

### Data

In [ ]:
df = pd.read_json(f"/content/drive/MyDrive/CheckThat Task2/{language}/clef2026_gpt4_o_mini_train_arabic.json")

In [ ]:
import random

final_training_data = []

for idx, item in df.iterrows():
    label = item['label'].lower()
    verdicts = [v.lower() for v in item['Verdict_list']]
    traces = [remove_label_pattern(t).split("Label:")[0] for t in item['Reasoning_traces']]

    # Identify indices of correct and incorrect traces
    correct_indices = [i for i, v in enumerate(verdicts) if v == label]
    incorrect_indices = [i for i, v in enumerate(verdicts) if v != label]

    # Skip if we can't form a pair
    if not correct_indices or not incorrect_indices:
        continue

    # Create pairs
    for c_idx in correct_indices:
        # Sample up to 3 incorrect traces to pair against to avoid data explosion
        sampled_incorrect = random.sample(incorrect_indices, min(3, len(incorrect_indices)))

        for inc_idx in sampled_incorrect:
            # 50% chance Correct trace is Candidate A
            if random.choice([True, False]):
                input_text = f"Claim: {item['claim']}\nCandidate A: {traces[c_idx]}\nCandidate B: {traces[inc_idx]}"
                target = 1.0
            # 50% chance Correct trace is Candidate B
            else:
                input_text = f"Claim: {item['claim']}\nCandidate A: {traces[inc_idx]}\nCandidate B: {traces[c_idx]}"
                target = 0.0

            final_training_data.append({
                "sample_id": f"{idx}_{c_idx}_{inc_idx}",
                "input_text": input_text,
                "Class": target
            })

print(f"Total Pairwise Samples Generated: {len(final_training_data)}")

# Save to JSONL for the TrainerModule
pd.DataFrame(final_training_data).to_json(
     f"/content/drive/MyDrive/CheckThat Task2/{language}/train_pairwise.jsonl",
     orient="records",
     lines=True,
     force_ascii=False
)

Total Pairwise Samples Generated: 23500


### Classes

In [ ]:
class PairwiseVerifierEvaluator:
    def __init__(self, model_path, tokenizer_path, base_model, quantization_config=None, token=None, device="cuda"):
        self.device = torch.device(device if torch.cuda.is_available() else "cpu")
        self.tokenizer = AutoTokenizer.from_pretrained(tokenizer_path, token=token)
        self.tokenizer.pad_token = self.tokenizer.eos_token

        # Load quantized base model
        self.model = CustomClassifier(
            base_model,
            use_lora=True,
            is_base_encoder=False,
            quantization_config=quantization_config,
            token=token,
            # Ensure quantization_config is accepted by CustomClassifier
        )

        # Load weights strictly to LoRA adapters and classifier head
        state_dict = torch.load(model_path, map_location=self.device)
        self.model.load_state_dict(state_dict, strict=False)
        self.model.to(self.device)
        self.model.eval()

    def encode_pair(self, claim, trace_a, trace_b, max_length=512): # Increased max_length for two traces
        text = f"Claim: {claim}\nCandidate A: {trace_a}\nCandidate B: {trace_b}"
        encoding = self.tokenizer(
            text,
            truncation=True,
            padding="max_length",
            max_length=max_length,
            return_tensors="pt",
        )
        return encoding["input_ids"].to(self.device), encoding["attention_mask"].to(self.device)

    def score_pair(self, claim, trace_a, trace_b):
        ids, mask = self.encode_pair(claim, trace_a, trace_b)
        with torch.no_grad():
            logit = float(self.model(ids, mask).item())
            # Positive logit = Model prefers A. Negative = Model prefers B.
            return logit

### Language

In [ ]:
language = "English"

In [ ]:
# ===== PATH =====
TRAIN_CSV = f"/content/drive/MyDrive/CheckThat Task2/{language}/train_pairwise.jsonl"
VAL_CSV   = test
# TEST_JSON = "/kaggle/input/YOUR_DATA/test.jsonl"

OUTPUT_DIR = f"/content/drive/MyDrive/CheckThat Task2/{language}"

# BASE_MODEL = "meta-llama/Llama-3.2-1B"
BASE_MODEL = "meta-llama/Llama-3.2-3B-Instruct"


MAX_LENGTH = 512
BATCH_SIZE = 4
EPOCHS = 3
LR = 1e-4


In [ ]:
!pip install --upgrade torchao transformers

In [ ]:
import torch
from transformers import AutoTokenizer, AutoModelForCausalLM, BitsAndBytesConfig

# # 1. Configure 4-bit quantization
# bnb_config = BitsAndBytesConfig(
#     load_in_4bit=True,
#     bnb_4bit_quant_type="nf4",
#     bnb_4bit_compute_dtype=torch.float16,
#     bnb_4bit_use_double_quant=True,
# )

tokenizer_llm = AutoTokenizer.from_pretrained(BASE_MODEL, token=HF_ACCESS_TOKEN)


### Training

In [ ]:
train_df = pd.read_json(TRAIN_CSV, lines = True)

train_df, val_df = train_test_split(
    train_df,
    test_size=0.2,
    stratify=train_df["Class"],
    random_state=42
)
# val_df = pd.read_json(VAL_CSV, lines = True)
tokenizer_llm = AutoTokenizer.from_pretrained(BASE_MODEL, token=HF_ACCESS_TOKEN)


train_dataset = TextDataset(train_df, tokenizer_llm, MAX_LENGTH)
val_dataset = TextDataset(val_df, tokenizer_llm, MAX_LENGTH)

train_loader = DataLoader(train_dataset, batch_size=BATCH_SIZE, shuffle=True)
val_loader = DataLoader(val_dataset, batch_size=BATCH_SIZE)

model = CustomClassifier(
    BASE_MODEL,
    use_lora=True,
    is_base_encoder=False,
    quantization_config=None,
    device_map="auto",
    token=HF_ACCESS_TOKEN
)

print_trainable_parameters(model)

trainer = TrainerModule(
    model,
    train_loader=train_loader,
    val_loader=val_loader,
    tokenizer = tokenizer_llm,
    epochs=EPOCHS,
    lr=LR,
    patience=2,
    output_dir=OUTPUT_DIR,
)

trainer.train()

Loading weights:   0%|          | 0/254 [00:00<?, ?it/s]

trainable params: 3214337 || all params: 3215964161 || trainable%: 0.10

Epoch 1/3


100%|██████████| 4700/4700 [17:30<00:00,  4.47it/s]


Train Loss: 0.2232
Train Acc: 0.8880
Val Loss: 0.0661
Val Acc: 0.9749

Epoch 2/3


100%|██████████| 4700/4700 [17:35<00:00,  4.45it/s]


Train Loss: 0.0340
Train Acc: 0.9883
Val Loss: 0.0281
Val Acc: 0.9909

Epoch 3/3


100%|██████████| 4700/4700 [17:35<00:00,  4.45it/s]


Train Loss: 0.0033
Train Acc: 0.9988
Val Loss: 0.0339
Val Acc: 0.9917


### Load Models

In [ ]:
pointwise_evaluator = VerifierEvaluator(
    model_path=f"/content/drive/MyDrive/CheckThat Task2/English/baseline/model",
    tokenizer_path=BASE_MODEL,
    base_model=BASE_MODEL,
    token=HF_ACCESS_TOKEN,
    use_decomp=True,
)

config.json:   0%|          | 0.00/843 [00:00<?, ?B/s]

tokenizer_config.json: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/301 [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/2.47G [00:00<?, ?B/s]

Loading weights:   0%|          | 0/146 [00:00<?, ?it/s]

ERROR:bitsandbytes.cextension:bitsandbytes library load error: libnvJitLink.so.13: cannot open shared object file: No such file or directory
Traceback (most recent call last):
  File "/usr/local/lib/python3.12/dist-packages/bitsandbytes/cextension.py", line 320, in <module>
    lib = get_native_library()
          ^^^^^^^^^^^^^^^^^^^^
  File "/usr/local/lib/python3.12/dist-packages/bitsandbytes/cextension.py", line 298, in get_native_library
    dll = ct.cdll.LoadLibrary(str(binary_path))
          ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "/usr/lib/python3.12/ctypes/__init__.py", line 460, in LoadLibrary
    return self._dlltype(name)
           ^^^^^^^^^^^^^^^^^^^
  File "/usr/lib/python3.12/ctypes/__init__.py", line 379, in __init__
    self._handle = _dlopen(self._name, mode)
                   ^^^^^^^^^^^^^^^^^^^^^^^^^
OSError: libnvJitLink.so.13: cannot open shared object file: No such file or directory


In [ ]:



pairwise_evaluator = PairwiseVerifierEvaluator(
    model_path=f"/content/drive/MyDrive/CheckThat Task2/English/llama3B/model_pairwise", # Path to your trained model
    tokenizer_path=BASE_MODEL,
    base_model=BASE_MODEL,
    # quantization_config=bnb_config,
    token=HF_ACCESS_TOKEN
    # pass bnb_config and token if using 3B model
)


Loading weights:   0%|          | 0/254 [00:00<?, ?it/s]

### Old Approach

In [ ]:
from tqdm import tqdm

with open(VAL_CSV, "r") as f:
    test_data = json.load(f)

predictions = []

evaluator = PairwiseVerifierEvaluator(
    model_path=f"{OUTPUT_DIR}/model", # Path to your trained model
    tokenizer_path=BASE_MODEL,
    base_model=BASE_MODEL,
    # quantization_config=bnb_config,
    token=HF_ACCESS_TOKEN
    # pass bnb_config and token if using 3B model
)

for idx, sample in enumerate(tqdm(test_data, desc="Evaluating Pairs")):
    claim = sample["claim"]
    verdicts = sample["Verdict_list"]
    traces = [remove_label_pattern(t).split("Label:")[0] for t in sample["Reasoning_traces"]]

    num_traces = len(traces)
    win_scores = [0.0] * num_traces

    # Round Robin: Compare every trace to every other trace
    for i in range(num_traces):
        for j in range(num_traces):
            if i == j:
                continue

            # Score Trace I vs Trace J
            score = evaluator.score_pair(claim, traces[i], traces[j])

            # If logit is positive, Trace I wins. Add the logit to its score.
            # If logit is negative, Trace J wins. We add absolute logit to J's score.
            if score > 0:
                win_scores[i] += score
            else:
                win_scores[j] += abs(score)

    # The BoN trace is the one with the highest total win score
    best_idx = np.argmax(np.array(win_scores))
    best_verdict = verdicts[best_idx]

    predictions.append({
        "query_id": idx,
        "Claim": claim,
        "Label": sample["label"],
        "Verdict_BoN": best_verdict,
        "BoN_Verdict_list": verdicts,
        "Reasoning_traces": traces,
        "score_list": win_scores, # This feeds perfectly into the organizer's IR metrics script
    })

# Save for the CLEF evaluation script
with open(f"/content/drive/MyDrive/CheckThat Task2/{language}/clef_predictions.json", "w") as fp:
    json.dump(predictions, fp, indent=4)

config.json:   0%|          | 0.00/843 [00:00<?, ?B/s]

tokenizer_config.json: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/301 [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/2.47G [00:00<?, ?B/s]

Loading weights:   0%|          | 0/146 [00:00<?, ?it/s]

Evaluating Pairs:   8%|▊         | 125/1600 [13:36<2:40:39,  6.54s/it]


KeyboardInterrupt: 

### BM25

In [ ]:
# ── Imports (add rank_bm25 if not already installed) ──────────────────────────
import math
import random
import json
import numpy as np
from tqdm import tqdm
from typing import List, Optional, Tuple

try:
    from rank_bm25 import BM25Okapi
    BM25_AVAILABLE = True
except ImportError:
    BM25_AVAILABLE = False
    print("rank-bm25 not installed. Run: !pip install rank-bm25")
    print("Falling back to random seeding (still faster than Round-Robin).")


# ─────────────────────────────────────────────────────────────────────────────
#  Stage 1: BM25 Seeder
#  Assigns a fast lexical relevance score to each trace before the bracket.
#  Higher score = better seed = placed higher/safer in the bracket.
#  Runs on CPU with no model calls — essentially free.
# ─────────────────────────────────────────────────────────────────────────────

def bm25_seed_scores(claim: str, traces: List[str]) -> List[float]:
    """
    Returns a normalised BM25 score in [0,1] for each trace.
    Higher = more lexically relevant to the claim = stronger seed.
    Falls back to equal scores if rank_bm25 is unavailable.
    """
    if not BM25_AVAILABLE or not traces:
        return [1.0] * len(traces)

    tokenised = [t.lower().split() for t in traces]
    bm25 = BM25Okapi(tokenised)
    query_tokens = claim.lower().split()
    raw = bm25.get_scores(query_tokens)           # numpy array

    lo, hi = raw.min(), raw.max()
    if hi - lo < 1e-9:
        return [1.0] * len(traces)                # all equal — neutral seeding

    return ((raw - lo) / (hi - lo)).tolist()

"""
Tournament Bracket Inference
=============================
Drop-in replacement for the Round-Robin inference loop in
Pairwise_train_verifier.ipynb.

Everything ABOVE this section stays identical:
  - CustomClassifier definition
  - PairwiseVerifierEvaluator definition
  - Training loop
  - `evaluator` object instantiation

Only replace the inference loop cell and the cells that follow it.

Key facts about your existing model that drive every design decision here:
  - score_pair() returns a RAW LOGIT (float), not a probability
  - Positive logit => Candidate A preferred; Negative => Candidate B preferred
  - Input format: "Claim: {claim}\nCandidate A: {trace_a}\nCandidate B: {trace_b}"
  - The evaluation script ranks traces by score_list descending (higher = better)

Round-Robin cost:  N*(N-1) calls  (ordered pairs, not combinations)
  → N=20:  380 model calls per claim

Tournament Bracket cost:  ~N-1 to N*log2(N) calls
  → N=20:  19 calls (winner only) to ~80 calls (full ranking)
  → ~5-20x speedup depending on N
"""


# ─────────────────────────────────────────────────────────────────────────────
#  Stage 2: Tournament Bracket
# ─────────────────────────────────────────────────────────────────────────────

def _interleave_seeds(seeded: list) -> list:
    """
    Standard bracket interleaving: maximally separates strong seeds so they
    can only meet in late rounds.
    Example for N=8: positions are assigned as [1,8,5,4,3,6,7,2]
    (strongest seeds are in opposite halves of the bracket).
    """
    size = len(seeded)
    if size <= 2:
        return seeded

    bracket = [None] * size

    def assign(seeds_slice, pos_slice):
        if not seeds_slice or not pos_slice:
            return
        if len(seeds_slice) == 1:
            bracket[pos_slice[0]] = seeds_slice[0]
            return
        mid = len(pos_slice) // 2
        bracket[pos_slice[0]] = seeds_slice[0]           # best seed → top slot
        bracket[pos_slice[mid]] = seeds_slice[1]         # 2nd seed → opposite half
        remaining = seeds_slice[2:]
        top_pos = pos_slice[1:mid]
        bot_pos = pos_slice[mid + 1:]
        half = len(remaining) // 2
        assign(remaining[:half], top_pos)
        assign(remaining[half:], bot_pos)

    assign(seeded, list(range(size)))
    return bracket

### Tournament Bracket

In [ ]:
def run_tournament(
    claim: str,
    traces: List[str],
    evaluator,                    # your PairwiseVerifierEvaluator instance
    seed_scores: Optional[List[float]] = None,
    max_length: int = 512,
) -> Tuple[List[int], List[float], int]:
    """
    Run a seeded single-elimination tournament over `traces`.

    Returns
    -------
    ranked_indices : List[int]
        Indices into `traces`, ordered best → worst.
    score_list : List[float]
        A score for every trace compatible with the CLEF evaluation script.
        Higher score = better rank. Assigned as (N - rank + 1) so the winner
        gets score N, runner-up gets N-1, etc.
        Ties within the same elimination round are broken by seed strength.
    n_comparisons : int
        Total model calls made (for logging / ablation reporting).
    """
    n = len(traces)
    if n == 0:
        return [], [], 0
    if n == 1:
        return [0], [1.0], 0

    indices = list(range(n))

    # ── Seed ordering (higher score = stronger seed = index 0 in sorted list) ──
    if seed_scores is not None:
        seeded_order = sorted(indices, key=lambda i: seed_scores[i], reverse=True)
    else:
        seeded_order = indices[:]
        random.shuffle(seeded_order)

    # ── Pad to next power of two with byes (None = auto-advance, no match played) ──
    bracket_size = 1 << math.ceil(math.log2(max(n, 2)))
    padded = seeded_order + [None] * (bracket_size - n)

    # ── Interleave so top seeds are in opposite bracket halves ──
    bracket = _interleave_seeds(padded)

    # ── Run rounds, recording which round each loser was eliminated in ──
    rounds_of_losers: List[List[int]] = []   # rounds_of_losers[0] = Round 1 losers
    n_comparisons = 0
    current = bracket

    while True:
        next_round = []
        round_losers = []

        for i in range(0, len(current), 2):
            a = current[i]
            b = current[i + 1] if i + 1 < len(current) else None

            if a is None and b is None:
                next_round.append(None)

            elif a is None:
                next_round.append(b)          # b gets a bye (free advance)

            elif b is None:
                next_round.append(a)          # a gets a bye (higher seed)

            else:
                # ── Real match ──────────────────────────────────────────────
                # score_pair returns the raw logit from your model.
                # Positive => Candidate A (traces[a]) preferred.
                # Negative => Candidate B (traces[b]) preferred.
                # Ties (exactly 0.0) broken by seed rank: earlier in seeded_order wins.
                logit = evaluator.score_pair(claim, traces[a], traces[b])
                n_comparisons += 1

                a_seed_rank = seeded_order.index(a)
                b_seed_rank = seeded_order.index(b)

                if logit > 0 or (logit == 0.0 and a_seed_rank < b_seed_rank):
                    next_round.append(a)
                    round_losers.append(b)
                else:
                    next_round.append(b)
                    round_losers.append(a)

        rounds_of_losers.append(round_losers)
        current = next_round

        real_survivors = [x for x in current if x is not None]
        if len(real_survivors) <= 1:
            winner = real_survivors[0] if real_survivors else None
            break

    # ── Build full ranking ────────────────────────────────────────────────────
    # Winner first, then losers from the most recent (final) round backwards.
    # Within each elimination round, sort by seed rank (stronger seed = higher rank).
    ranked: List[int] = []

    if winner is not None:
        ranked.append(winner)

    for round_losers in reversed(rounds_of_losers):
        # Sort losers in this round: stronger seed (lower index in seeded_order) ranks higher
        sorted_losers = sorted(
            round_losers,
            key=lambda idx: seeded_order.index(idx) if idx in seeded_order else 9999
        )
        ranked.extend(sorted_losers)

    # ── Build score_list compatible with the CLEF evaluation script ───────────
    # The eval script does: sorted(..., key=lambda i: score_list[i], reverse=True)
    # We assign score = (N - rank_position) so winner gets N, last gets 1.
    score_list = [0.0] * n
    for rank_pos, trace_idx in enumerate(ranked):
        score_list[trace_idx] = float(n - rank_pos)

    return ranked, score_list, n_comparisons

In [ ]:

# ─────────────────────────────────────────────────────────────────────────────
#  Main inference loop  (replaces the Round-Robin cell in the notebook)
# ─────────────────────────────────────────────────────────────────────────────

with open(VAL_CSV, "r") as f:
    test_data = json.load(f)

predictions = []
total_comparisons = 0
total_rr_equivalent = 0

evaluator = PairwiseVerifierEvaluator(
    model_path=f"/content/drive/MyDrive/CheckThat Task2/English/llama3B/model_pairwise", # Path to your trained model
    tokenizer_path=BASE_MODEL,
    base_model=BASE_MODEL,
    # quantization_config=bnb_config,
    token=HF_ACCESS_TOKEN
    # pass bnb_config and token if using 3B model
)


for idx, sample in enumerate(tqdm(test_data, desc="Tournament Ranking")):
    claim   = sample["claim"]
    verdicts = sample["Verdict_list"]
    traces  = [remove_label_pattern(t).split("Label:")[0] for t in sample["Reasoning_traces"]]
    n       = len(traces)

    # ── Stage 1: BM25 seeding (no GPU, no model calls) ──
    seeds = bm25_seed_scores(claim, traces)

    # ── Stage 2: Tournament bracket ──
    ranked_indices, score_list, n_comp = run_tournament(
        claim=claim,
        traces=traces,
        evaluator=evaluator,
        seed_scores=seeds,
        max_length=512,            # must match what PairwiseVerifierEvaluator uses
    )

    total_comparisons   += n_comp
    total_rr_equivalent += n * (n - 1)         # what Round-Robin would have done

    # ── Pick best verdict (winner of the tournament) ──
    best_idx     = ranked_indices[0]
    best_verdict = verdicts[best_idx]

    predictions.append({
        "query_id":        idx,
        "Claim":           claim,
        # "Label":           sample["label"],
        "Verdict_BoN":     best_verdict,
        "BoN_Verdict_list": verdicts,
        "Reasoning_traces": traces,
        "score_list":      score_list,      # compatible with CLEF eval script
    })

    # ── Save predictions ──────────────────────────────────────────────────────────
output_path = f"/content/drive/MyDrive/CheckThat Task2/{language}/llama3B/clef_predictions_tournament_3b.json"
with open(output_path, "w") as fp:
    json.dump(predictions, fp, indent=4)

Loading weights:   0%|          | 0/254 [00:00<?, ?it/s]

Tournament Ranking: 100%|██████████| 2558/2558 [46:10<00:00,  1.08s/it]


In [ ]:

# ── Efficiency report ─────────────────────────────────────────────────────────
reduction = 100 * (1 - total_comparisons / total_rr_equivalent) if total_rr_equivalent > 0 else 0
print(f"\n{'='*55}")
print(f"  Tournament Bracket — Efficiency Report")
print(f"{'='*55}")
print(f"  Claims evaluated     : {len(test_data)}")
print(f"  Total model calls    : {total_comparisons:,}")
print(f"  Round-Robin would use: {total_rr_equivalent:,}")
print(f"  Reduction            : {reduction:.1f}%")
print(f"  Avg calls per claim  : {total_comparisons / max(len(test_data), 1):.1f}")
print(f"  Predictions saved to : {output_path}")
print(f"{'='*55}")

# ── Then run the Evaluation cell as-is, pointing INPUT_PATH at the new file ──
INPUT_PATH = output_path


  Tournament Bracket — Efficiency Report
  Claims evaluated     : 652
  Total model calls    : 12,388
  Round-Robin would use: 247,760
  Reduction            : 95.0%
  Avg calls per claim  : 19.0
  Predictions saved to : /content/drive/MyDrive/CheckThat Task2/Arabic/clef_predictions_tournament.json


### PageRank

In [ ]:
"""
PageRank Augmentation for Tournament Bracket Ranking
=====================================================
Augments the existing tournament_inference.py with graph-based centrality.

Core idea
---------
Every match the tournament runs produces:
  - a winner
  - a loser
  - the model's logit confidence (magnitude = how decisive the win was)

This is exactly a weighted directed graph:
  edge  winner → loser  with weight = |logit|

PageRank on that graph asks: "which trace won against the strongest opponents?"
Unlike the bracket's integer tier scores (N, N-1, N-2…), PageRank gives
continuous scores that distinguish traces that won convincingly vs. barely.

The final score_list fed to the CLEF evaluation script is a convex combination:
  score_list[i] = α * bracket_score[i] + β * pagerank_score[i]
  where α + β = 1

Setting α=1 recovers pure bracket; α=0 gives pure PageRank.
This lets you ablate all three approaches in one run.

What stays the same
-------------------
- CustomClassifier          — unchanged
- PairwiseVerifierEvaluator — unchanged
- bm25_seed_scores          — unchanged
- The outer inference loop  — replace run_tournament() with run_tournament_with_pagerank()

New dependency
--------------
  networkx (for PageRank)
  !pip install networkx  (usually already installed in Colab)
"""

import math
import random
import json
import numpy as np
from tqdm import tqdm
from typing import List, Optional, Tuple, Dict

try:
    import networkx as nx
    NX_AVAILABLE = True
except ImportError:
    NX_AVAILABLE = False
    print("networkx not installed. Run: !pip install networkx")




# ─────────────────────────────────────────────────────────────────────────────
#  PageRank scoring
# ─────────────────────────────────────────────────────────────────────────────

def compute_pagerank(
    n_traces: int,
    match_log: List[Dict],
    damping: float = 0.85,
    max_iter: int = 100,
) -> List[float]:
    """
    Build a directed weighted graph from match results and run PageRank.

    Graph semantics
    ---------------
    Edge: winner → loser
    Weight: |logit|  (higher = more decisive win = stronger evidence)

    A trace that wins convincingly against already-strong traces will receive
    a high PageRank score, capturing "won against good opponents" — exactly
    the signal bracket integer tiers miss.

    Args
    ----
    n_traces  : total number of traces (nodes 0..n_traces-1)
    match_log : list of dicts with keys 'winner', 'loser', 'logit_magnitude'
    damping   : PageRank damping factor (standard 0.85)
    max_iter  : maximum power-iteration steps

    Returns
    -------
    List of PageRank scores (length n_traces), normalised to [0, 1].
    Traces that never participated in a match get the uniform prior (1/N).
    """
    if not NX_AVAILABLE:
        # Fallback: uniform scores — no PageRank applied
        return [1.0 / n_traces] * n_traces

    G = nx.DiGraph()
    G.add_nodes_from(range(n_traces))

    for match in match_log:
        w = match["winner"]
        l = match["loser"]
        weight = max(match["logit_magnitude"], 1e-6)   # avoid zero-weight edges

        if G.has_edge(w, l):
            G[w][l]["weight"] += weight                # accumulate if multiple matches
        else:
            G.add_edge(w, l, weight=weight)

    # Nodes with no outgoing edges (all losers who never won) get a self-loop
    # at negligible weight to keep the graph strongly connected for PageRank.
    for node in G.nodes():
        if G.out_degree(node) == 0:
            G.add_edge(node, node, weight=1e-9)

    pr = nx.pagerank(G, alpha=damping, max_iter=max_iter, weight="weight")

    scores = [pr.get(i, 1.0 / n_traces) for i in range(n_traces)]

    # Normalise to [0, 1]
    lo, hi = min(scores), max(scores)
    if hi - lo < 1e-12:
        return [1.0] * n_traces
    return [(s - lo) / (hi - lo) for s in scores]


# ─────────────────────────────────────────────────────────────────────────────
#  Tournament with match logging
# ─────────────────────────────────────────────────────────────────────────────

def run_tournament_with_pagerank(
    claim: str,
    traces: List[str],
    evaluator,
    seed_scores: Optional[List[float]] = None,
    alpha: float = 0.5,            # weight for bracket score  (0.0 = pure PageRank)
    beta: float = 0.5,             # weight for PageRank score (1.0 = pure bracket)
    damping: float = 0.85,
    max_length: int = 512,
) -> Tuple[List[int], List[float], int, List[Dict]]:
    """
    Tournament bracket with PageRank-augmented score fusion.

    Parameters
    ----------
    alpha : float
        Weight of the bracket's integer tier scores in the final score_list.
    beta : float
        Weight of PageRank scores.  alpha + beta should equal 1.0.

    Returns
    -------
    ranked_indices : best → worst ordering of trace indices
    score_list     : fused score per trace, compatible with CLEF eval script
    n_comparisons  : total model calls made
    match_log      : raw match data (useful for ablation analysis)

    Ablation table
    --------------
    alpha=1.0, beta=0.0  →  pure tournament bracket  (your current system)
    alpha=0.0, beta=1.0  →  pure PageRank
    alpha=0.5, beta=0.5  →  equal fusion  (recommended starting point)
    """
    n = len(traces)
    if n == 0:
        return [], [], 0, []
    if n == 1:
        return [0], [1.0], 0, []

    indices = list(range(n))
    if seed_scores is not None:
        seeded_order = sorted(indices, key=lambda i: seed_scores[i], reverse=True)
    else:
        seeded_order = indices[:]
        random.shuffle(seeded_order)

    bracket_size = 1 << math.ceil(math.log2(max(n, 2)))
    padded = seeded_order + [None] * (bracket_size - n)
    bracket = _interleave_seeds(padded)

    rounds_of_losers: List[List[int]] = []
    match_log: List[Dict] = []          # ← NEW: log every match result
    n_comparisons = 0
    current = bracket

    while True:
        next_round = []
        round_losers = []

        for i in range(0, len(current), 2):
            a = current[i]
            b = current[i + 1] if i + 1 < len(current) else None

            if a is None and b is None:
                next_round.append(None)

            elif a is None:
                next_round.append(b)

            elif b is None:
                next_round.append(a)

            else:
                logit = evaluator.score_pair(claim, traces[a], traces[b])
                n_comparisons += 1

                a_seed_rank = seeded_order.index(a)
                b_seed_rank = seeded_order.index(b)

                if logit > 0 or (logit == 0.0 and a_seed_rank < b_seed_rank):
                    winner, loser = a, b
                else:
                    winner, loser = b, a

                next_round.append(winner)
                round_losers.append(loser)

                # ── Log this match for the PageRank graph ──────────────────
                match_log.append({
                    "winner":          winner,
                    "loser":           loser,
                    "logit_magnitude": abs(logit),
                    "logit_raw":       logit,           # keep for analysis
                    "round":           len(rounds_of_losers),
                })

        rounds_of_losers.append(round_losers)
        current = next_round

        real_survivors = [x for x in current if x is not None]
        if len(real_survivors) <= 1:
            winner = real_survivors[0] if real_survivors else None
            break

    # ── Build bracket ranking ─────────────────────────────────────────────────
    ranked: List[int] = []
    if winner is not None:
        ranked.append(winner)
    for rl in reversed(rounds_of_losers):
        sorted_losers = sorted(
            rl, key=lambda idx: seeded_order.index(idx) if idx in seeded_order else 9999
        )
        ranked.extend(sorted_losers)

    # Bracket scores: winner=N, runner-up=N-1, …, last=1
    bracket_scores = [0.0] * n
    for rank_pos, trace_idx in enumerate(ranked):
        bracket_scores[trace_idx] = float(n - rank_pos)

    # Normalise bracket scores to [0, 1]
    bmax = max(bracket_scores)
    bmin = min(bracket_scores)
    if bmax - bmin > 1e-9:
        bracket_norm = [(s - bmin) / (bmax - bmin) for s in bracket_scores]
    else:
        bracket_norm = [1.0] * n

    # ── PageRank scores ───────────────────────────────────────────────────────
    pr_scores = compute_pagerank(n, match_log, damping=damping)

    # ── Fused score_list ──────────────────────────────────────────────────────
    # Both components are now in [0,1]. The convex combination keeps the
    # result in [0,1] and compatible with the CLEF evaluation script
    # (which only needs relative ordering, not absolute values).
    score_list = [
        alpha * bracket_norm[i] + beta * pr_scores[i]
        for i in range(n)
    ]

    # Derive final ranked order from the fused scores
    ranked_indices = sorted(range(n), key=lambda i: score_list[i], reverse=True)

    return ranked_indices, score_list, n_comparisons, match_log

In [ ]:

# ─────────────────────────────────────────────────────────────────────────────
#  Inference loop  (drop-in replacement for the tournament_inference.py loop)
# ─────────────────────────────────────────────────────────────────────────────
# Paste this cell into your notebook, replacing the existing inference loop.
# `evaluator`, `VAL_CSV`, `language`, `remove_label_pattern` are all unchanged.

# ── CONFIG — tune these for ablation ──────────────────────────────────────────
ALPHA = 0.7   # bracket score weight  (set 1.0 to replicate pure-bracket baseline)
BETA  = 0.3    # PageRank score weight (set 1.0 for pure-PageRank ablation)
DAMPING = 0.85 # standard PageRank damping factor
# ─────────────────────────────────────────────────────────────────────────────

with open(VAL_CSV, "r") as f:
    test_data = json.load(f)

predictions = []
all_match_logs = []          # keep for post-hoc graph analysis if needed
total_comparisons = 0
total_rr_equivalent = 0

evaluator = PairwiseVerifierEvaluator(
    model_path=f"/content/drive/MyDrive/CheckThat Task2/English/pairwise_tournament/model", # Path to your trained model
    tokenizer_path=BASE_MODEL,
    base_model=BASE_MODEL,
    # quantization_config=bnb_config,
    token=HF_ACCESS_TOKEN
    # pass bnb_config and token if using 3B model
)

for idx, sample in enumerate(tqdm(test_data, desc=f"PageRank-augmented ranking (α={ALPHA}, β={BETA})")):
    claim    = sample["claim"]
    verdicts = sample["Verdict_list"]
    traces   = [remove_label_pattern(t).split("Label:")[0] for t in sample["Reasoning_traces"]]
    n        = len(traces)

    # Stage 1: BM25 seeding
    seeds = bm25_seed_scores(claim, traces)

    # Stage 2: Tournament + PageRank fusion
    ranked_indices, score_list, n_comp, match_log = run_tournament_with_pagerank(
        claim=claim,
        traces=traces,
        evaluator=evaluator,
        seed_scores=None,
        alpha=ALPHA,
        beta=BETA,
        damping=DAMPING,
    )

    total_comparisons   += n_comp
    total_rr_equivalent += n * (n - 1)
    all_match_logs.append(match_log)

    best_idx     = ranked_indices[0]
    best_verdict = verdicts[best_idx]

    predictions.append({
        "query_id":         idx,
        "Claim":            claim,
        "Label":            sample["label"],
        "Verdict_BoN":      best_verdict,
        "BoN_Verdict_list": verdicts,
        "Reasoning_traces": traces,
        "score_list":       score_list,     # CLEF-compatible
    })

# ── Save ──────────────────────────────────────────────────────────────────────
tag = f"alpha{int(ALPHA*10)}_beta{int(BETA*10)}"
output_path = f"/content/drive/MyDrive/CheckThat Task2/{language}/clef_predictions_pagerank_{tag}.json"

with open(output_path, "w") as fp:
    json.dump(predictions, fp, indent=4)

reduction = 100 * (1 - total_comparisons / total_rr_equivalent) if total_rr_equivalent > 0 else 0

print(f"\n{'='*55}")
print(f"  PageRank-augmented Tournament — Report")
print(f"{'='*55}")
print(f"  α (bracket weight) : {ALPHA}")
print(f"  β (PageRank weight): {BETA}")
print(f"  Claims evaluated   : {len(test_data)}")
print(f"  Total model calls  : {total_comparisons:,}")
print(f"  Round-Robin equiv. : {total_rr_equivalent:,}")
print(f"  Reduction          : {reduction:.1f}%")
print(f"  Avg calls / claim  : {total_comparisons / max(len(test_data), 1):.1f}")
print(f"  Predictions saved  : {output_path}")
print(f"{'='*55}")
print(f"\n  Ablation: re-run with ALPHA=1.0, BETA=0.0 for pure bracket baseline")
print(f"  Ablation: re-run with ALPHA=0.0, BETA=1.0 for pure PageRank baseline")

# Point the evaluation cell at the new file
INPUT_PATH = output_path

config.json:   0%|          | 0.00/843 [00:00<?, ?B/s]

tokenizer_config.json: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

KeyboardInterrupt: 

In [ ]:
# ─────────────────────────────────────────────────────────────────────────────
#  Optional: per-claim graph diagnostics
#  Run this cell after inference to inspect which traces dominate the graph
# ─────────────────────────────────────────────────────────────────────────────

def print_claim_graph_stats(claim_idx: int, predictions: list, all_match_logs: list):
    """
    Prints the match graph and PageRank distribution for a single claim.
    Useful for understanding why the ranker chose a particular top trace.
    """
    if not NX_AVAILABLE:
        print("networkx not available — skipping diagnostics.")
        return

    entry    = predictions[claim_idx]
    log      = all_match_logs[claim_idx]
    n        = len(entry["Reasoning_traces"])
    verdicts = entry["BoN_Verdict_list"]
    label    = entry["Label"].lower()
    scores   = entry["score_list"]

    print(f"\n=== Claim {claim_idx}: {entry['Claim'][:80]} ===")
    print(f"Ground truth: {label}")
    print(f"\nMatch log ({len(log)} comparisons):")
    for m in log:
        v_w = verdicts[m["winner"]].lower()
        v_l = verdicts[m["loser"]].lower()
        mark = "✓" if v_w == label else "✗"
        print(f"  {mark} trace_{m['winner']}({v_w}) beat trace_{m['loser']}({v_l})  "
              f"logit={m['logit_raw']:+.3f}  round={m['round']}")

    print(f"\nFused score ranking (top 5):")
    ranked = sorted(range(n), key=lambda i: scores[i], reverse=True)
    for pos, idx in enumerate(ranked[:5], 1):
        v = verdicts[idx].lower()
        mark = "✓" if v == label else "✗"
        print(f"  #{pos}  trace_{idx}  verdict={v}  score={scores[idx]:.4f}  {mark}")


# Example usage (uncomment to run):
print_claim_graph_stats(claim_idx=0, predictions=predictions, all_match_logs=all_match_logs)

### PRP

In [ ]:
"""
PRP — Pairwise Ranking Prompting
=================================
Drop-in replacement for the Round-Robin inference loop in
Pairwise_train_verifier.ipynb (Cell 24).

Everything above this cell is unchanged:
  - CustomClassifier
  - PairwiseVerifierEvaluator  (and the `evaluator` instance)
  - Training loop
  - remove_label_pattern / VAL_CSV / language / OUTPUT_DIR

The Problem: Positional Bias
-----------------------------
Your model was trained with 50/50 A/B randomisation, so it saw both orderings
during training. But at inference each call only sees one ordering, and LLMs
are known to have a residual preference for whichever candidate appears first —
even after balanced training.

Concretely, consider one comparison:
  score_pair(claim, good_trace, bad_trace)  →  +0.3  (good wins, correct)
  score_pair(claim, bad_trace, good_trace)  →  +0.5  (bad wins as A, WRONG)

The two calls disagree. Round-Robin makes 380 such one-sided calls and silently
accumulates the noise into win_scores.

The PRP Fix
-----------
For every (i, j) pair, run the comparison in BOTH orderings and combine:

  forward_logit  = score_pair(claim, trace_i, trace_j)
        → positive means i preferred as A
  backward_logit = score_pair(claim, trace_j, trace_i)
        → positive means j preferred as A, so negate to get "i preferred"

  debiased_score = (forward_logit - backward_logit) / 2

  If debiased_score > 0: trace_i wins
  Margin = |debiased_score| added to win_scores[i]

This is mathematically equivalent to: "i truly beats j only if the model
prefers i in BOTH orderings, or prefers it strongly enough in forward to
outweigh any backward bias."

Cost: exactly 2× the model calls of one-sided Round-Robin.
      N*(N-1) one-sided  →  N*(N-1) two-sided  (same count, different pairing)
      Because one-sided already calls (i,j) AND (j,i) — so PRP doesn't add
      calls, it just *uses* both calls together instead of independently.

      Wait — re-reading Cell 24: the loop is `for i in range(N): for j in range(N): if i!=j`
      That's already N*(N-1) calls covering BOTH (i,j) and (j,i).
      PRP restructures these into N*(N-1)/2 *debiased* pairs — same total calls,
      better use of the signal.

Integration Options
-------------------
A) PRP-RoundRobin  : same O(N^2) call budget, fully debiased  ← implemented here
B) PRP-Tournament  : plug prp_score_pair() into tournament_inference.py
                     instead of score_pair() — halves calls AND debiases
"""

import json
import numpy as np
from tqdm import tqdm


# ─────────────────────────────────────────────────────────────────────────────
#  Core PRP function  — wraps your existing evaluator.score_pair()
# ─────────────────────────────────────────────────────────────────────────────

def prp_score(evaluator, claim: str, trace_i: str, trace_j: str) -> float:
    """
    Debiased pairwise score: how much trace_i is preferred over trace_j.

    Runs the comparison in both orderings and cancels positional bias:

        forward  = score_pair(claim, i, j)   # positive → model prefers i as A
        backward = score_pair(claim, j, i)   # positive → model prefers j as A
                                             # negated  → model prefers i

        debiased = (forward - backward) / 2

    Returns
    -------
    float
        Positive  → trace_i is preferred (debiased).
        Negative  → trace_j is preferred (debiased).
        Magnitude → confidence of the preference.

    Model calls: exactly 2 per (i, j) pair.
    """
    forward  = evaluator.score_pair(claim, trace_i, trace_j)
    backward = evaluator.score_pair(claim, trace_j, trace_i)

    # backward positive means j is preferred as A.
    # Negating flips it to "i is preferred" perspective, matching forward.
    debiased = (forward - backward) / 2.0
    return debiased


#### Round Robin

In [ ]:
# ─────────────────────────────────────────────────────────────────────────────
#  Option A: PRP-Round-Robin
#  Same structure as Cell 24 but uses prp_score() per pair.
#  Total model calls: N*(N-1)  — identical to original, zero extra cost.
# ─────────────────────────────────────────────────────────────────────────────

with open(VAL_CSV, "r") as f:
    test_data = json.load(f)

predictions = []
total_forward_calls = 0

for idx, sample in enumerate(tqdm(test_data, desc="PRP Round-Robin")):
    claim    = sample["claim"]
    verdicts = sample["Verdict_list"]
    traces   = [remove_label_pattern(t).split("Label:")[0]
                for t in sample["Reasoning_traces"]]
    n        = len(traces)

    win_scores = [0.0] * n

    # Iterate over unique pairs only — (i, j) and (j, i) are handled
    # inside prp_score() as the forward/backward calls.
    for i in range(n):
        for j in range(i + 1, n):          # upper triangle only
            debiased = prp_score(evaluator, claim, traces[i], traces[j])
            total_forward_calls += 2       # prp_score makes 2 model calls

            if debiased > 0:
                win_scores[i] += debiased          # i wins, add confidence
            elif debiased < 0:
                win_scores[j] += abs(debiased)     # j wins, add confidence
            # exact tie (debiased == 0): neither gets credit

    best_idx     = int(np.argmax(win_scores))
    best_verdict = verdicts[best_idx]

    predictions.append({
        "query_id":         idx,
        "Claim":            claim,
        "Label":            sample["label"],
        "Verdict_BoN":      best_verdict,
        "BoN_Verdict_list": verdicts,
        "Reasoning_traces": traces,
        "score_list":       win_scores,    # CLEF-compatible: higher = better rank
    })

output_path = (
    f"/content/drive/MyDrive/CheckThat Task2/{language}"
    f"/clef_predictions_prp_roundrobin.json"
)
with open(output_path, "w") as fp:
    json.dump(predictions, fp, indent=4)

n_sample   = len(test_data)
avg_traces = sum(len(s["Reasoning_traces"]) for s in test_data) / max(n_sample, 1)
orig_calls = int(avg_traces * (avg_traces - 1)) * n_sample   # original one-sided RR

print(f"\n{'='*55}")
print(f"  PRP Round-Robin — Summary")
print(f"{'='*55}")
print(f"  Claims evaluated      : {n_sample}")
print(f"  Avg traces per claim  : {avg_traces:.1f}")
print(f"  Total model calls     : {total_forward_calls:,}")
print(f"  Original RR calls     : {orig_calls:,}  (same budget)")
print(f"  Debiasing             : both orderings fused per pair")
print(f"  Predictions saved to  : {output_path}")
print(f"{'='*55}")

# Point the evaluation cell at this file
INPUT_PATH = output_path


# ─────────────────────────────────────────────────────────────────────────────
#  Option B: PRP-Tournament  (paste into tournament_inference.py instead)
#  Replace `evaluator.score_pair(...)` with `prp_score(evaluator, ...)` in
#  run_tournament(). Cuts calls by ~95% AND debiases each comparison.
#
#  In tournament_inference.py, find this block inside run_tournament():
#
#      logit = evaluator.score_pair(claim, traces[a], traces[b])
#      n_comparisons += 1
#
#  Replace with:
#
#      logit = prp_score(evaluator, claim, traces[a], traces[b])
#      n_comparisons += 2   # prp_score uses 2 calls
#
#  Everything else — win/loss logic, seeding, loser tracking — stays identical.
#  The debiased logit has the same sign contract: positive = a wins.
# ─────────────────────────────────────────────────────────────────────────────


#### Tournament

In [ ]:
def run_tournament_prp(
    claim: str,
    traces: List[str],
    evaluator,                    # your PairwiseVerifierEvaluator instance
    seed_scores: Optional[List[float]] = None,
    max_length: int = 512,
) -> Tuple[List[int], List[float], int]:
    """
    Run a seeded single-elimination tournament over `traces`.

    Returns
    -------
    ranked_indices : List[int]
        Indices into `traces`, ordered best → worst.
    score_list : List[float]
        A score for every trace compatible with the CLEF evaluation script.
        Higher score = better rank. Assigned as (N - rank + 1) so the winner
        gets score N, runner-up gets N-1, etc.
        Ties within the same elimination round are broken by seed strength.
    n_comparisons : int
        Total model calls made (for logging / ablation reporting).
    """
    n = len(traces)
    if n == 0:
        return [], [], 0
    if n == 1:
        return [0], [1.0], 0

    indices = list(range(n))

    # ── Seed ordering (higher score = stronger seed = index 0 in sorted list) ──
    if seed_scores is not None:
        seeded_order = sorted(indices, key=lambda i: seed_scores[i], reverse=True)
    else:
        seeded_order = indices[:]
        random.shuffle(seeded_order)

    # ── Pad to next power of two with byes (None = auto-advance, no match played) ──
    bracket_size = 1 << math.ceil(math.log2(max(n, 2)))
    padded = seeded_order + [None] * (bracket_size - n)

    # ── Interleave so top seeds are in opposite bracket halves ──
    bracket = _interleave_seeds(padded)

    # ── Run rounds, recording which round each loser was eliminated in ──
    rounds_of_losers: List[List[int]] = []   # rounds_of_losers[0] = Round 1 losers
    n_comparisons = 0
    current = bracket

    while True:
        next_round = []
        round_losers = []

        for i in range(0, len(current), 2):
            a = current[i]
            b = current[i + 1] if i + 1 < len(current) else None

            if a is None and b is None:
                next_round.append(None)

            elif a is None:
                next_round.append(b)          # b gets a bye (free advance)

            elif b is None:
                next_round.append(a)          # a gets a bye (higher seed)

            else:
                # ── Real match ──────────────────────────────────────────────
                # score_pair returns the raw logit from your model.
                # Positive => Candidate A (traces[a]) preferred.
                # Negative => Candidate B (traces[b]) preferred.
                # Ties (exactly 0.0) broken by seed rank: earlier in seeded_order wins.
                logit = prp_score(evaluator, claim, traces[a], traces[b])
                n_comparisons += 2

                a_seed_rank = seeded_order.index(a)
                b_seed_rank = seeded_order.index(b)

                if logit > 0 or (logit == 0.0 and a_seed_rank < b_seed_rank):
                    next_round.append(a)
                    round_losers.append(b)
                else:
                    next_round.append(b)
                    round_losers.append(a)

        rounds_of_losers.append(round_losers)
        current = next_round

        real_survivors = [x for x in current if x is not None]
        if len(real_survivors) <= 1:
            winner = real_survivors[0] if real_survivors else None
            break

    # ── Build full ranking ────────────────────────────────────────────────────
    # Winner first, then losers from the most recent (final) round backwards.
    # Within each elimination round, sort by seed rank (stronger seed = higher rank).
    ranked: List[int] = []

    if winner is not None:
        ranked.append(winner)

    for round_losers in reversed(rounds_of_losers):
        # Sort losers in this round: stronger seed (lower index in seeded_order) ranks higher
        sorted_losers = sorted(
            round_losers,
            key=lambda idx: seeded_order.index(idx) if idx in seeded_order else 9999
        )
        ranked.extend(sorted_losers)

    # ── Build score_list compatible with the CLEF evaluation script ───────────
    # The eval script does: sorted(..., key=lambda i: score_list[i], reverse=True)
    # We assign score = (N - rank_position) so winner gets N, last gets 1.
    score_list = [0.0] * n
    for rank_pos, trace_idx in enumerate(ranked):
        score_list[trace_idx] = float(n - rank_pos)

    return ranked, score_list, n_comparisons

In [ ]:

# ─────────────────────────────────────────────────────────────────────────────
#  Main inference loop
# ─────────────────────────────────────────────────────────────────────────────

with open(VAL_CSV, "r") as f:
    test_data = json.load(f)

predictions = []
total_comparisons = 0
total_rr_equivalent = 0

evaluator = PairwiseVerifierEvaluator(
    model_path=f"/content/drive/MyDrive/CheckThat Task2/English/llama3B/model_pairwise", # Path to your trained model
    tokenizer_path=BASE_MODEL,
    base_model=BASE_MODEL,
    # quantization_config=bnb_config,
    token=HF_ACCESS_TOKEN
    # pass bnb_config and token if using 3B model
)


for idx, sample in enumerate(tqdm(test_data, desc="Tournament Ranking")):
    claim   = sample["claim"]
    verdicts = sample["Verdict_list"]
    traces  = [remove_label_pattern(t).split("Label:")[0] for t in sample["Reasoning_traces"]]
    n       = len(traces)

    # ── Stage 1: BM25 seeding (no GPU, no model calls) ──
    seeds = bm25_seed_scores(claim, traces)

    # ── Stage 2: Tournament bracket ──
    ranked_indices, score_list, n_comp = run_tournament_prp(
        claim=claim,
        traces=traces,
        evaluator=evaluator,
        seed_scores=seeds,
        max_length=512,            # must match what PairwiseVerifierEvaluator uses
    )

    total_comparisons   += n_comp
    total_rr_equivalent += n * (n - 1)         # what Round-Robin would have done

    # ── Pick best verdict (winner of the tournament) ──
    best_idx     = ranked_indices[0]
    best_verdict = verdicts[best_idx]

    predictions.append({
        "query_id":        idx,
        "Claim":           claim,
        # "Label":           sample["label"],
        "Verdict_BoN":     best_verdict,
        "BoN_Verdict_list": verdicts,
        "Reasoning_traces": traces,
        "score_list":      score_list,      # compatible with CLEF eval script
    })

    # ── Save predictions ──────────────────────────────────────────────────────────
output_path = f"/content/drive/MyDrive/CheckThat Task2/{language}/prp/clef_predictions.json"
with open(output_path, "w") as fp:
    json.dump(predictions, fp, indent=4)

Loading weights:   0%|          | 0/254 [00:00<?, ?it/s]

Tournament Ranking:   7%|▋         | 167/2558 [06:07<1:27:40,  2.20s/it]


KeyboardInterrupt: 

# PrePAIR

In [ ]:
# ═══════════════════════════════════════════════════════════════════════════════
#  PrePAIR — Corrected Implementation
#  ─────────────────────────────────────────────────────────────────────────────
#  Paste this as a single cell in PrePAIR_train_verifier.ipynb, replacing
#  Cells 30-33. Cells 34 (ablation guide) and 37 (evaluation) are unchanged.
#
#  Prerequisites that must already be instantiated above this cell:
#    - `pointwise_evaluator` : VerifierEvaluator  (from PrePAIR notebook)
#    - `pairwise_evaluator`  : PairwiseVerifierEvaluator  (from Pairwise notebook)
#    - `language`            : str  e.g. "English"
#    - `VAL_CSV`             : str  path to validation JSON
#    - `remove_label_pattern`: function  (already defined in Cell 15)
#
#  Two-model design
#  ─────────────────
#  PrePAIR's critique step asks: "how good is this single trace?"
#  That is exactly what the POINTWISE model was trained to answer.
#  Using it here is correct — it scores (claim, verdict, justification)
#  independently, which is its training objective.
#
#  The PAIRWISE model then receives the pre-computed scorecards and makes
#  the final comparison call — also exactly what it was trained to do,
#  but now anchored to explicit intermediate evidence rather than raw text.
#
#  Why the previous generate()-based approach was wrong
#  ─────────────────────────────────────────────────────
#  CustomClassifier wraps AutoModel (encoder-only, no causal LM head).
#  Calling .generate() on it either fails outright or produces incoherent
#  output because the model has no text generation head. The pointwise
#  model's .score() method already does exactly what we need: a forward
#  pass that returns a scalar quality logit for a single trace.
# ═══════════════════════════════════════════════════════════════════════════════

import math
import json
import numpy as np
from tqdm import tqdm


# ───────────────────────────────────────────────────────────────────────────────
#  Language config  (unchanged from previous version)
# ───────────────────────────────────────────────────────────────────────────────

def get_language_config(language: str) -> dict:
    lang = language.lower().strip()
    if lang == "arabic":
        return {
            "lang_code": "arabic",
            "simplified_criteria": True,
            "verdicts_to_score": ["true", "false", "conflicting"],
        }
    elif lang == "spanish":
        return {
            "lang_code": "spanish",
            "simplified_criteria": False,
            "verdicts_to_score": ["true", "false", "conflicting"],
        }
    else:  # english
        return {
            "lang_code": "english",
            "simplified_criteria": False,
            "verdicts_to_score": ["true", "false", "conflicting"],
        }


In [ ]:

# ───────────────────────────────────────────────────────────────────────────────
#  Stage 1: Critique via the POINTWISE model
#  ─────────────────────────────────────────
#  VerifierEvaluator.score(claim, questions, verdict, justification) returns a
#  raw logit (float). Higher = the model thinks this (verdict, justification)
#  pair is a good answer for this claim.
#
#  We call it three times per trace — once per candidate verdict label —
#  and record all three logits as the scorecard. This tells the pairwise
#  model not just "is this trace good?" but "what does the pointwise model
#  think the correct label is, and how confident is it?"
#
#  For Arabic (simplified), we call it once with the trace's own verdict
#  to keep the prompt load manageable.
# ───────────────────────────────────────────────────────────────────────────────

def build_scorecard(
    pointwise_evaluator,        # VerifierEvaluator instance
    claim: str,
    trace: str,
    own_verdict: str,           # the verdict this trace actually predicts
    lang_config: dict,
) -> dict:
    """
    Score a single trace using the pointwise model.

    Returns a dict with:
      "own_verdict"        : the trace's predicted verdict
      "own_verdict_score"  : pointwise logit for (claim, own_verdict, trace)
      "true_score"         : pointwise logit when verdict="true"   [non-Arabic]
      "false_score"        : pointwise logit when verdict="false"  [non-Arabic]
      "conflicting_score"  : pointwise logit when verdict="conflicting" [non-Arabic]
      "max_verdict"        : which verdict the pointwise model scores highest
      "score_spread"       : max_score - min_score (confidence proxy)
    """
    questions = ""  # VerifierEvaluator.encode_input accepts questions=""

    # Always score with the trace's own verdict
    own_score = pointwise_evaluator.score(claim, questions, own_verdict, trace)

    scorecard = {
        "own_verdict":       own_verdict,
        "own_verdict_score": own_score,
    }

    if lang_config["simplified_criteria"]:
        # Arabic: single score only — keeps token load down
        scorecard["max_verdict"]  = own_verdict
        scorecard["score_spread"] = 0.0
        return scorecard

    # English / Spanish: score all three verdict hypotheses
    verdict_scores = {}
    for v in lang_config["verdicts_to_score"]:
        verdict_scores[v] = pointwise_evaluator.score(claim, questions, v, trace)

    best_verdict = max(verdict_scores, key=verdict_scores.get)
    scores_list  = list(verdict_scores.values())

    scorecard.update({
        "true_score":        verdict_scores["true"],
        "false_score":       verdict_scores["false"],
        "conflicting_score": verdict_scores["conflicting"],
        "max_verdict":       best_verdict,
        "score_spread":      max(scores_list) - min(scores_list),
    })
    return scorecard


def format_scorecard_for_prompt(scorecard: dict, label: str, simplified: bool) -> str:
    """
    Renders the scorecard as a compact block for injection into the
    pairwise comparison prompt.
    """
    lines = [f"[{label}]"]
    lines.append(f"  predicted_verdict: {scorecard['own_verdict']}")
    lines.append(f"  pointwise_score:   {scorecard['own_verdict_score']:.4f}")

    if not simplified:
        lines.append(f"  true_score:        {scorecard['true_score']:.4f}")
        lines.append(f"  false_score:       {scorecard['false_score']:.4f}")
        lines.append(f"  conflicting_score: {scorecard['conflicting_score']:.4f}")
        lines.append(f"  best_verdict:      {scorecard['max_verdict']}")
        lines.append(f"  score_spread:      {scorecard['score_spread']:.4f}")

    return "\n".join(lines)



# ───────────────────────────────────────────────────────────────────────────────
#  Patch PairwiseVerifierEvaluator to accept an optional prompt override
#  ─────────────────────────────────────────────────────────────────────────────
#  Your current score_pair() builds the prompt internally and does not
#  accept external prompts. We monkey-patch it here so you don't need to
#  edit the original class definition.
#
#  This patch:
#    1. Saves the original method as _score_pair_original
#    2. Replaces score_pair with a version that accepts _override_prompt
#    3. Falls through to the original for all other arguments unchanged
#
#  If you prefer to edit the class directly, add this to score_pair():
#    def score_pair(self, claim, trace_a, trace_b, _override_prompt=None):
#        if _override_prompt is not None:
#            text = _override_prompt
#        else:
#            text = f"Claim: {claim}\nCandidate A: {trace_a}\nCandidate B: {trace_b}"
#        # ... rest unchanged
# ───────────────────────────────────────────────────────────────────────────────

import types

def _patched_score_pair(self, claim, trace_a, trace_b, _override_prompt=None):
    """
    Patched score_pair that accepts an optional prompt override.
    When _override_prompt is None, behaviour is identical to the original.
    """
    if _override_prompt is not None:
        text = _override_prompt
    else:
        text = f"Claim: {claim}\nCandidate A: {trace_a}\nCandidate B: {trace_b}"

    # Reuse the original tokenization and forward pass logic
    encoding = self.tokenizer(
        text,
        return_tensors="pt",
        max_length=MAX_LENGTH,   # use whatever max_length the instance has
        truncation=True,
        padding=False,
    )
    input_ids      = encoding["input_ids"].to(self.device)
    attention_mask = encoding["attention_mask"].to(self.device)

    import torch
    with torch.no_grad():
        logit = self.model(input_ids, attention_mask)

    return float(logit.item())

# Apply the patch to your existing pairwise_evaluator instance.
# (pairwise_evaluator must already exist in scope above this cell.)
pairwise_evaluator._score_pair_original = pairwise_evaluator.score_pair
pairwise_evaluator.score_pair = types.MethodType(_patched_score_pair, pairwise_evaluator)
print("score_pair patched successfully.")


score_pair patched successfully.


In [ ]:

# ───────────────────────────────────────────────────────────────────────────────
#  Stage 2: Comparison with scorecard  (PAIRWISE model)
# ───────────────────────────────────────────────────────────────────────────────

def build_comparison_prompt(
    claim: str,
    trace_a: str,
    trace_b: str,
    scorecard_a: dict,
    scorecard_b: dict,
    simplified: bool,
) -> str:
    """
    Builds the enriched pairwise prompt.
    The pairwise model sees the pointwise scorecards BEFORE the raw traces,
    giving it explicit intermediate evidence to anchor the comparison.
    """
    card_a = format_scorecard_for_prompt(scorecard_a, "Trace A", simplified)
    card_b = format_scorecard_for_prompt(scorecard_b, "Trace B", simplified)

    return (
        f"Claim: {claim}\n\n"
        f"Pre-evaluation (pointwise model scores):\n"
        f"{card_a}\n\n"
        f"{card_b}\n\n"
        f"Candidate A: {trace_a}\n"
        f"Candidate B: {trace_b}"
    )


def compare_prepair(
    pairwise_evaluator,
    claim: str,
    trace_a: str,
    trace_b: str,
    scorecard_a: dict,
    scorecard_b: dict,
    simplified: bool,
    use_prp: bool = True,
) -> float:
    """
    Runs the scorecard-enriched pairwise comparison.

    If use_prp=True (recommended): runs in both directions and averages
    to cancel positional bias (PRP debiasing).

    Returns: positive logit => trace_a preferred, negative => trace_b preferred.
    Model calls: 2 if use_prp=True, 1 if False.
    """
    prompt_fwd = build_comparison_prompt(
        claim, trace_a, trace_b, scorecard_a, scorecard_b, simplified
    )
    logit_fwd = pairwise_evaluator.score_pair(
        claim, trace_a, trace_b, _override_prompt=prompt_fwd
    )

    if not use_prp:
        return logit_fwd

    # PRP reverse pass: swap A/B and their scorecards
    prompt_rev = build_comparison_prompt(
        claim, trace_b, trace_a, scorecard_b, scorecard_a, simplified
    )
    logit_rev = pairwise_evaluator.score_pair(
        claim, trace_b, trace_a, _override_prompt=prompt_rev
    )

    # Average cancels positional bias ε:
    # forward  = true_pref + ε
    # reverse  = -(true_pref + ε) → negated → true_pref - ε
    # average  = true_pref
    return (logit_fwd - logit_rev) / 2.0


# ───────────────────────────────────────────────────────────────────────────────
#  Tournament bracket helpers  (inlined — no external import needed)
# ───────────────────────────────────────────────────────────────────────────────

def _interleave_seeds(seeded: list) -> list:
    size = len(seeded)
    if size <= 2:
        return seeded
    bracket = [None] * size

    def assign(seeds_slice, pos_slice):
        if not seeds_slice or not pos_slice:
            return
        if len(seeds_slice) == 1:
            bracket[pos_slice[0]] = seeds_slice[0]
            return
        mid = len(pos_slice) // 2
        bracket[pos_slice[0]]   = seeds_slice[0]
        bracket[pos_slice[mid]] = seeds_slice[1]
        remaining = seeds_slice[2:]
        assign(remaining[:len(remaining)//2], pos_slice[1:mid])
        assign(remaining[len(remaining)//2:], pos_slice[mid + 1:])

    assign(seeded, list(range(size)))
    return bracket


def run_tournament_prepair(
    claim, traces, verdicts,
    pointwise_evaluator, pairwise_evaluator,
    lang_config, scorecard_cache,
    use_prp=True,
):
    """
    Tournament bracket where:
      - Seeding order = pointwise score on own_verdict (highest = seed #1)
      - Match winner  = pairwise model with PRP + scorecard enrichment

    Returns (ranked_indices, score_list, n_comparison_calls)
    """
    n = len(traces)
    if n == 0:
        return [], [], 0
    if n == 1:
        return [0], [1.0], 0

    simplified = lang_config["simplified_criteria"]

    # Seed by pointwise own-verdict score: best scorer gets the top seed
    seeded_order = sorted(
        range(n),
        key=lambda i: scorecard_cache[i]["own_verdict_score"],
        reverse=True,
    )

    bracket_size = 1 << math.ceil(math.log2(max(n, 2)))
    padded       = seeded_order + [None] * (bracket_size - n)
    bracket      = _interleave_seeds(padded)

    rounds_of_losers  = []
    n_comparison_calls = 0
    current            = bracket

    while True:
        next_round   = []
        round_losers = []

        for i in range(0, len(current), 2):
            a = current[i]
            b = current[i + 1] if i + 1 < len(current) else None

            if a is None and b is None:
                next_round.append(None)
            elif a is None:
                next_round.append(b)
            elif b is None:
                next_round.append(a)
            else:
                logit = compare_prepair(
                    pairwise_evaluator,
                    claim,
                    traces[a], traces[b],
                    scorecard_cache[a], scorecard_cache[b],
                    simplified, use_prp,
                )
                n_comparison_calls += 2 if use_prp else 1

                a_seed = seeded_order.index(a)
                b_seed = seeded_order.index(b)
                winner, loser = (
                    (a, b) if (logit > 0 or (logit == 0 and a_seed < b_seed))
                    else (b, a)
                )
                next_round.append(winner)
                round_losers.append(loser)

        rounds_of_losers.append(round_losers)
        current = next_round
        real    = [x for x in current if x is not None]
        if len(real) <= 1:
            winner_node = real[0] if real else None
            break

    # Build full ranking: winner first, then losers latest-round first
    ranked = []
    if winner_node is not None:
        ranked.append(winner_node)
    for rl in reversed(rounds_of_losers):
        ranked.extend(sorted(
            rl,
            key=lambda idx: seeded_order.index(idx) if idx in seeded_order else 9999,
        ))

    # score_list compatible with CLEF eval script (higher = better rank)
    score_list = [0.0] * n
    for rank_pos, trace_idx in enumerate(ranked):
        score_list[trace_idx] = float(n - rank_pos)

    return ranked, score_list, n_comparison_calls


In [ ]:
language = "English"

In [ ]:

# ───────────────────────────────────────────────────────────────────────────────
#  CONFIG
# ───────────────────────────────────────────────────────────────────────────────

USE_PRP        = False   # PRP debiasing in pairwise comparison calls
USE_TOURNAMENT = True   # tournament bracket (recommended); False = Round-Robin

lang_config = get_language_config(language)
print(f"PrePAIR config: lang={lang_config['lang_code']}, "
      f"simplified={lang_config['simplified_criteria']}, "
      f"use_prp={USE_PRP}, use_tournament={USE_TOURNAMENT}")


# ───────────────────────────────────────────────────────────────────────────────
#  Inference loop
# ───────────────────────────────────────────────────────────────────────────────

with open(VAL_CSV, "r") as f:
    test_data = json.load(f)

predictions            = []
total_critique_calls   = 0
total_comparison_calls = 0
total_rr_equivalent    = 0
simplified             = lang_config["simplified_criteria"]
n_verdict_calls        = 1 if simplified else len(lang_config["verdicts_to_score"])





for idx, sample in enumerate(tqdm(test_data, desc=f"PrePAIR [{lang_config['lang_code']}]")):
    claim    = sample["claim"]
    verdicts = sample["Verdict_list"]
    traces   = [
        remove_label_pattern(t).split("Label:")[0]
        for t in sample["Reasoning_traces"]
    ]
    n = len(traces)
    total_rr_equivalent += n * (n - 1)

    # ── Stage 1: Build scorecard for every trace using the POINTWISE model ───
    # Each scorecard call costs n_verdict_calls model calls.
    # Scorecards are cached and reused for all comparisons involving that trace.
    scorecard_cache = {}
    for i, trace in enumerate(traces):
        own_verdict = verdicts[i].lower()
        scorecard_cache[i] = build_scorecard(
            pointwise_evaluator, claim, trace, own_verdict, lang_config
        )
        total_critique_calls += n_verdict_calls

    # ── Stage 2: Rank traces using the PAIRWISE model + scorecards ──────────
    if USE_TOURNAMENT:
        ranked, score_list, n_comp = run_tournament_prepair(
            claim, traces, verdicts,
            pointwise_evaluator, pairwise_evaluator,
            lang_config, scorecard_cache,
            use_prp=USE_PRP,
        )
        total_comparison_calls += n_comp

    else:
        # Round-Robin upper triangle with PrePAIR + optional PRP
        win_scores = [0.0] * n
        for i in range(n):
            for j in range(i + 1, n):
                logit = compare_prepair(
                    pairwise_evaluator,
                    claim,
                    traces[i], traces[j],
                    scorecard_cache[i], scorecard_cache[j],
                    simplified, USE_PRP,
                )
                total_comparison_calls += 2 if USE_PRP else 1
                if logit > 0:
                    win_scores[i] += logit
                else:
                    win_scores[j] += abs(logit)

        score_list = win_scores
        ranked     = sorted(range(n), key=lambda x: score_list[x], reverse=True)

    best_idx     = ranked[0]
    best_verdict = verdicts[best_idx]

    predictions.append({
        "query_id":         idx,
        "Claim":            claim,
        # "Label":            sample["label"],
        "Verdict_BoN":      best_verdict,
        "BoN_Verdict_list": verdicts,
        "Reasoning_traces": traces,
        "score_list":       score_list,
    })

# ── Save ─────────────────────────────────────────────────────────────────────
tag = (
    f"prepair"
    f"{'_prp' if USE_PRP else ''}"
    f"{'_tournament' if USE_TOURNAMENT else '_rr'}"
    f"_{lang_config['lang_code']}"
)
output_path = (
    f"/content/drive/MyDrive/CheckThat Task2/{language}/"
    f"clef_predictions_{tag}.json"
)
with open(output_path, "w") as fp:
    json.dump(predictions, fp, indent=4)

total_model_calls = total_critique_calls + total_comparison_calls
reduction = (
    100 * (1 - total_model_calls / total_rr_equivalent)
    if total_rr_equivalent else 0
)

print(f"\n{'='*58}")
print(f"  PrePAIR [{lang_config['lang_code']}] — Run report")
print(f"{'='*58}")
print(f"  Pointwise calls (critique)  : {total_critique_calls:,}  "
      f"({n_verdict_calls} per trace × {sum(len(s['Reasoning_traces']) for s in test_data)} traces)")
print(f"  Pairwise calls (comparison) : {total_comparison_calls:,}")
print(f"  Total model calls           : {total_model_calls:,}")
print(f"  Round-Robin equivalent      : {total_rr_equivalent:,}")
print(f"  Reduction vs Round-Robin    : {reduction:.1f}%")
print(f"  Saved → {output_path}")
print(f"{'='*58}")

# Point the evaluation cell at this output
INPUT_PATH = output_path

PrePAIR config: lang=english, simplified=False, use_prp=False, use_tournament=True


PrePAIR [english]: 100%|██████████| 2558/2558 [2:35:35<00:00,  3.65s/it]



  PrePAIR [english] — Run report
  Pointwise calls (critique)  : 153,480  (3 per trace × 51160 traces)
  Pairwise calls (comparison) : 48,602
  Total model calls           : 202,082
  Round-Robin equivalent      : 972,040
  Reduction vs Round-Robin    : 79.2%
  Saved → /content/drive/MyDrive/CheckThat Task2/English/clef_predictions_prepair_tournament_english.json


In [ ]:
# ─────────────────────────────────────────────────────────────────────────────
#  Ablation guide
#  ─────────────────────────────────────────────────────────────────────────────
#
#  Run these configurations in order to isolate PrePAIR's contribution:
#
#  1. Baseline (your current best):
#       USE_PRP=True, USE_TOURNAMENT=True, no PrePAIR
#       → clef_predictions_prp_tournament_{lang}.json
#
#  2. PrePAIR alone (no PRP):
#       USE_PRP=False, USE_TOURNAMENT=True, PrePAIR=True
#       → isolates PrePAIR's criteria effect from PRP's debiasing
#
#  3. PrePAIR + PRP (recommended):
#       USE_PRP=True, USE_TOURNAMENT=True, PrePAIR=True
#       → expected best result for EN and ES
#
#  4. Arabic specific — try simplified vs full criteria:
#       lang_config = get_language_config("arabic")  # already simplified
#       vs manually setting simplified_criteria=False to test full criteria
#
#  5. Criteria ablation (optional, for the paper):
#       Remove one criterion at a time from lang_config["criteria"]
#       to identify which criterion contributes most per language

# DPO + Bradley Terry Reward Modelling

In [ ]:
"""
DPO + Bradley-Terry Training
==============================
Drop-in replacement for the pairwise training cells in
Pairwise_train_verifier.ipynb.

Everything that stays identical
---------------------------------
  Cell 1–13  : imports, language, data loading, helpers
  Cell 14    : CustomClassifier          — unchanged
  Cell 15    : TextDataset               — unchanged (reused for val split)
  Cell 20    : PairwiseVerifierEvaluator — unchanged (inference is identical)
  Cell 24+   : all inference cells       — unchanged

What changes
-------------
  Cell 19 : data pipeline  → BTPairDataset (keeps same JSONL schema)
  Cell 16 : TrainerModule  → BTTrainerModule (replaces BCEWithLogitsLoss)
  Cell 23 : training call  → uses BTTrainerModule

Why Bradley-Terry fixes your current model
-------------------------------------------
Your current BCE loss treats each pairwise example independently:
  loss = BCE(σ(logit), label)   ← label ∈ {0, 1}

This allows cyclic inconsistencies: the model can prefer A>B, B>C, C>A
across different examples in the same epoch, because no loss term
connects them. The model has no incentive to be globally consistent.

Bradley-Terry models the probability that trace A is preferred over
trace B as a function of latent reward scores:
  P(A ≻ B) = σ(r_A - r_B)      ← σ = sigmoid

The reward r_i is the model's scalar logit for "Claim: ...\nCandidate A: {i}".
This is exactly what CustomClassifier already outputs — we just need to
score each candidate separately and compute the difference.

The BT loss for a pair where A is the winner:
  loss = -log σ(r_winner - r_loser)
       = log(1 + exp(r_loser - r_winner))
       = softplus(r_loser - r_winner)

This is the DPO/RLHF reward model objective applied to your setting.
It forces the model to assign globally consistent rewards: if A>B and
B>C, then A>C follows from the reward magnitudes, not from luck.

Key implementation detail
--------------------------
BCE operates on ONE forward pass per pair (both candidates concatenated).
BT requires TWO forward passes per pair:
  1. encode "Claim: ...\nCandidate A: {winner_trace}"   → r_winner
  2. encode "Claim: ...\nCandidate A: {loser_trace}"    → r_loser
  3. loss = softplus(r_loser - r_winner)

We reuse CustomClassifier with num_labels=1. The output is the reward.
The classifier head is the same; only the loss function changes.

Data format
-----------
The existing Cell 19 stores:
  {"input_text": "Claim: ...\nCandidate A: winner\nCandidate B: loser",
   "Class": 1.0}   (or 0.0 when loser/winner are swapped)

BTPairDataset reconstructs (claim, winner_trace, loser_trace) from this
by parsing the input_text and checking Class. No re-generation needed —
the JSONL files you already have are sufficient.

Inference compatibility
-----------------------
PairwiseVerifierEvaluator.score_pair() is unchanged. After BT training,
the model's logit for "Claim: ...\nCandidate A: X\nCandidate B: Y" is
NOT a BT reward — it's still a concatenated-pair logit. But the model's
internal representations have been shaped by BT-consistent gradients,
so the concatenated-pair logit is more reliable.

For a pure BT inference mode (score each trace independently and
subtract), see BTVerifierEvaluator at the bottom of this file.
"""

import os
import re
import json
import random
import datetime
import numpy as np
import pandas as pd
from tqdm import tqdm

import torch
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
from torch.optim import AdamW
from transformers import AutoTokenizer, get_cosine_schedule_with_warmup
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score
import evaluate


# ─────────────────────────────────────────────────────────────────────────────
#  Cell 19 replacement: BT-aware data pipeline
#  Parses existing JSONL into (claim, winner_trace, loser_trace) triples.
#  No new data files needed — reads train_pairwise.jsonl as-is.
# ─────────────────────────────────────────────────────────────────────────────

def parse_pairwise_jsonl_to_bt(jsonl_path: str) -> pd.DataFrame:
    """
    Reads the existing train_pairwise.jsonl (produced by Cell 19) and
    reconstructs (claim, winner_trace, loser_trace) for BT training.

    The existing format stores:
      input_text = "Claim: {claim}\nCandidate A: {trace_a}\nCandidate B: {trace_b}"
      Class      = 1.0 if A is the correct (winner) trace, else 0.0

    Returns a DataFrame with columns:
      claim, winner_trace, loser_trace, sample_id
    """
    df = pd.read_json(jsonl_path, lines=True)
    records = []

    for _, row in df.iterrows():
        text = row["input_text"]
        is_a_winner = float(row["Class"]) == 1.0

        # Parse claim / candidate A / candidate B from the input text
        # Format is always "Claim: ...\nCandidate A: ...\nCandidate B: ..."
        claim_match = re.search(
            r"Claim:\s*(.*?)\nCandidate A:", text, re.DOTALL
        )
        a_match = re.search(
            r"Candidate A:\s*(.*?)\nCandidate B:", text, re.DOTALL
        )
        b_match = re.search(
            r"Candidate B:\s*(.*?)$", text, re.DOTALL
        )

        if not (claim_match and a_match and b_match):
            continue

        claim   = claim_match.group(1).strip()
        trace_a = a_match.group(1).strip()
        trace_b = b_match.group(1).strip()

        winner_trace = trace_a if is_a_winner else trace_b
        loser_trace  = trace_b if is_a_winner else trace_a

        records.append({
            "sample_id":    row.get("sample_id", ""),
            "claim":        claim,
            "winner_trace": winner_trace,
            "loser_trace":  loser_trace,
        })

    print(f"Parsed {len(records)} BT training pairs from {jsonl_path}")
    return pd.DataFrame(records)


# ─────────────────────────────────────────────────────────────────────────────
#  BTPairDataset
#  Each item yields two encoded sequences: winner and loser.
#  The DataLoader returns batched tensors for both.
# ─────────────────────────────────────────────────────────────────────────────

class BTPairDataset(Dataset):
    """
    Encodes each training pair as two separate sequences:
      winner_text = "Claim: {claim}\nCandidate A: {winner_trace}"
      loser_text  = "Claim: {claim}\nCandidate A: {loser_trace}"

    The BT loss needs r_winner and r_loser as separate forward passes.
    We encode both here so the DataLoader can batch them efficiently.

    Note: we always put the candidate in the A slot. This is intentional —
    BT rewards are absolute (per-trace), not relative to the other candidate.
    Putting both in the same slot removes positional bias from the reward
    signal by construction.
    """

    def __init__(self, dataframe: pd.DataFrame, tokenizer, max_length: int = 256):
        self.tokenizer  = tokenizer
        self.tokenizer.pad_token = self.tokenizer.eos_token
        self.max_length = max_length

        self.claims        = dataframe["claim"].tolist()
        self.winner_traces = dataframe["winner_trace"].tolist()
        self.loser_traces  = dataframe["loser_trace"].tolist()

    def __len__(self):
        return len(self.claims)

    def _encode(self, claim: str, trace: str) -> dict:
        text = f"Claim: {claim}\nCandidate A: {trace}"
        enc  = self.tokenizer(
            text,
            truncation=True,
            padding="max_length",
            max_length=self.max_length,
            return_tensors="pt",
        )
        return {k: v.squeeze(0) for k, v in enc.items()}

    def __getitem__(self, idx):
        winner_enc = self._encode(self.claims[idx], self.winner_traces[idx])
        loser_enc  = self._encode(self.claims[idx], self.loser_traces[idx])
        return {
            "winner_input_ids":      winner_enc["input_ids"],
            "winner_attention_mask": winner_enc["attention_mask"],
            "loser_input_ids":       loser_enc["input_ids"],
            "loser_attention_mask":  loser_enc["attention_mask"],
        }


# ─────────────────────────────────────────────────────────────────────────────
#  Bradley-Terry loss
# ─────────────────────────────────────────────────────────────────────────────

def bradley_terry_loss(r_winner: torch.Tensor, r_loser: torch.Tensor) -> torch.Tensor:
    """
    BT loss for a batch of (winner, loser) reward pairs.

    P(winner ≻ loser) = σ(r_winner - r_loser)
    loss = -log P = softplus(r_loser - r_winner)
         = log(1 + exp(r_loser - r_winner))

    Using F.softplus is numerically stable (avoids overflow for large logits).

    Args
    ----
    r_winner : shape (B,) — reward scores for winner traces
    r_loser  : shape (B,) — reward scores for loser traces

    Returns
    -------
    Scalar mean loss over the batch.
    """
    return F.softplus(r_loser - r_winner).mean()


# ─────────────────────────────────────────────────────────────────────────────
#  BTTrainerModule  (replaces TrainerModule / Cell 16)
# ─────────────────────────────────────────────────────────────────────────────

class BTTrainerModule:
    """
    Trains CustomClassifier with Bradley-Terry loss instead of BCE.

    Differences from the original TrainerModule
    --------------------------------------------
    1. loss_fn: BCEWithLogitsLoss → bradley_terry_loss
    2. forward pass: 1 per batch item → 2 (winner + loser separately)
    3. accuracy metric: binary BCE accuracy → BT accuracy
       (proportion of pairs where r_winner > r_loser)
    4. val_loader: uses BTPairDataset, not TextDataset

    Everything else (optimizer, scheduler, saving) is identical.
    """

    def __init__(
        self,
        model,
        train_loader: DataLoader,
        val_loader: DataLoader,
        tokenizer,
        epochs: int,
        lr: float,
        output_dir: str,
        patience: int = 2,
    ):
        self.device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
        self.model  = model.to(self.device)

        self.train_loader = train_loader
        self.val_loader   = val_loader
        self.tokenizer    = tokenizer
        self.epochs       = epochs
        self.patience     = patience
        self.output_dir   = output_dir

        self.optimizer = AdamW(model.parameters(), lr=lr, eps=1e-8)

        total_steps  = len(train_loader) * epochs
        warmup_steps = int(0.05 * total_steps)
        self.scheduler = get_cosine_schedule_with_warmup(
            self.optimizer, warmup_steps, total_steps
        )

        os.makedirs(output_dir, exist_ok=True)

        # Early stopping state
        self.best_val_loss = float("inf")
        self.epochs_no_improve = 0

    def _forward_pair(self, batch):
        """
        Two forward passes per batch: winner traces, then loser traces.
        Returns (r_winner, r_loser) as 1-D tensors of shape (B,).
        """
        r_winner = self.model(
            batch["winner_input_ids"].to(self.device),
            batch["winner_attention_mask"].to(self.device),
        ).squeeze(1)   # (B, 1) → (B,)

        r_loser = self.model(
            batch["loser_input_ids"].to(self.device),
            batch["loser_attention_mask"].to(self.device),
        ).squeeze(1)

        return r_winner, r_loser

    def train(self):
        for epoch in range(self.epochs):
            print(f"\nEpoch {epoch + 1}/{self.epochs}")
            self.model.train()

            total_loss = 0.0
            total_acc  = 0.0

            for batch in tqdm(self.train_loader, desc="train"):
                self.optimizer.zero_grad()

                r_winner, r_loser = self._forward_pair(batch)
                loss = bradley_terry_loss(r_winner, r_loser)

                loss.backward()
                # Gradient clipping — same as original (implicit via scheduler)
                torch.nn.utils.clip_grad_norm_(self.model.parameters(), 1.0)
                self.optimizer.step()
                self.scheduler.step()

                total_loss += loss.item()
                # BT accuracy: fraction of pairs where model correctly ranks winner > loser
                total_acc += (r_winner > r_loser).float().mean().item()

            n = len(self.train_loader)
            print(f"  Train BT loss : {total_loss / n:.4f}")
            print(f"  Train BT acc  : {total_acc  / n:.4f}  "
                  f"(% pairs ranked correctly)")

            val_loss = self._evaluate(epoch)

            # Early stopping
            if val_loss < self.best_val_loss:
                self.best_val_loss     = val_loss
                self.epochs_no_improve = 0
            else:
                self.epochs_no_improve += 1
                if self.epochs_no_improve >= self.patience:
                    print(f"  Early stopping after {epoch + 1} epochs.")
                    break

    def _evaluate(self, epoch: int) -> float:
        self.model.eval()
        total_loss = 0.0
        total_acc  = 0.0

        with torch.no_grad():
            for batch in tqdm(self.val_loader, desc="val  "):
                r_winner, r_loser = self._forward_pair(batch)
                loss = bradley_terry_loss(r_winner, r_loser)
                total_loss += loss.item()
                total_acc  += (r_winner > r_loser).float().mean().item()

        n = len(self.val_loader)
        val_loss = total_loss / n
        print(f"  Val  BT loss  : {val_loss:.4f}")
        print(f"  Val  BT acc   : {total_acc / n:.4f}")

        # Save — identical to original TrainerModule
        self.tokenizer.save_pretrained(self.output_dir)
        torch.save(
            self.model.state_dict(),
            os.path.join(self.output_dir, "model_bt"),
        )
        return val_loss


In [ ]:


# ─────────────────────────────────────────────────────────────────────────────
#  Cell 23 replacement: training call
#  Paste this into your notebook in place of Cell 23.
#  All variables (BASE_MODEL, OUTPUT_DIR, HF_ACCESS_TOKEN, language) unchanged.
# ─────────────────────────────────────────────────────────────────────────────

TRAIN_PAIRWISE_JSONL = (
    f"/content/drive/MyDrive/CheckThat Task2/{language}/train_pairwise.jsonl"
)

# ── Parse existing JSONL into BT format ──────────────────────────────────────
bt_df = parse_pairwise_jsonl_to_bt(TRAIN_PAIRWISE_JSONL)

train_bt_df, val_bt_df = train_test_split(
    bt_df, test_size=0.2, random_state=42
)

tokenizer_llm = AutoTokenizer.from_pretrained(BASE_MODEL, token=HF_ACCESS_TOKEN)

# Max length 256 per sequence (not 512) because each sequence now encodes
# only ONE trace (not a pair), so 256 is sufficient and halves memory use.
BT_MAX_LENGTH = 256

train_bt_dataset = BTPairDataset(train_bt_df, tokenizer_llm, BT_MAX_LENGTH)
val_bt_dataset   = BTPairDataset(val_bt_df,   tokenizer_llm, BT_MAX_LENGTH)

# Batch size can be 2× original because sequences are half the length
BT_BATCH_SIZE = 8   # was 4 in original — adjust down if OOM

train_bt_loader = DataLoader(train_bt_dataset, batch_size=BT_BATCH_SIZE, shuffle=True)
val_bt_loader   = DataLoader(val_bt_dataset,   batch_size=BT_BATCH_SIZE)

model_bt = CustomClassifier(
    BASE_MODEL,
    num_labels=1,
    use_lora=True,
    is_base_encoder=False,
    device_map="auto",
    token=HF_ACCESS_TOKEN,
)
print_trainable_parameters(model_bt)

trainer_bt = BTTrainerModule(
    model=model_bt,
    train_loader=train_bt_loader,
    val_loader=val_bt_loader,
    tokenizer=tokenizer_llm,
    epochs=EPOCHS,
    lr=LR,
    output_dir=OUTPUT_DIR,
    patience=2,
)

trainer_bt.train()



Parsed 19765 BT training pairs from /content/drive/MyDrive/CheckThat Task2/Spanish/train_pairwise.jsonl


Loading weights:   0%|          | 0/254 [00:00<?, ?it/s]

trainable params: 3214337 || all params: 3215964161 || trainable%: 0.10

Epoch 1/3


train: 100%|██████████| 1977/1977 [14:11<00:00,  2.32it/s]


  Train BT loss : 0.2697
  Train BT acc  : 0.8933  (% pairs ranked correctly)


...

  Val  BT loss  : 0.0501
  Val  BT acc   : 0.9879

Epoch 3/3


train: 100%|██████████| 1977/1977 [14:11<00:00,  2.32it/s]


  Train BT loss : 0.0030
  Train BT acc  : 0.9994  (% pairs ranked correctly)


val  : 100%|██████████| 495/495 [01:36<00:00,  5.11it/s]


  Val  BT loss  : 0.0315
  Val  BT acc   : 0.9912


In [ ]:

# ─────────────────────────────────────────────────────────────────────────────
#  BTVerifierEvaluator
#  Pure BT inference: scores each trace independently and subtracts.
#  This is the theoretically correct way to use a BT-trained model.
#
#  Use this INSTEAD OF PairwiseVerifierEvaluator when running ablations
#  to measure the pure BT effect. The existing PairwiseVerifierEvaluator
#  also works (it still scores concatenated pairs) but does not use the
#  BT reward interpretation.
# ─────────────────────────────────────────────────────────────────────────────

class BTVerifierEvaluator:
    """
    BT-mode evaluator: scores each trace independently.

    score_pair(claim, trace_a, trace_b) = r(claim, trace_a) - r(claim, trace_b)

    where r(claim, trace) = model("Claim: {claim}\nCandidate A: {trace}")

    Positive = trace_a preferred. Negative = trace_b preferred.
    Drop-in replacement for PairwiseVerifierEvaluator.score_pair().

    Caching
    -------
    For Round-Robin or tournament, each trace is compared against many
    others. Without caching, trace_i is encoded N-1 times. The reward
    cache computes r(trace_i) once and reuses it across all its matchups,
    reducing encoding calls from N*(N-1) to N.

    Usage
    -----
    evaluator = BTVerifierEvaluator(
        model_path=f"{OUTPUT_DIR}/model_bt",
        tokenizer_path=BASE_MODEL,
        base_model=BASE_MODEL,
        token=HF_ACCESS_TOKEN,
    )

    # Compute all rewards for a claim's traces upfront (recommended)
    rewards = evaluator.score_all(claim, traces)
    # rewards[i] = r(claim, traces[i]) — use directly as score_list

    # Or use as drop-in for score_pair (no caching)
    logit = evaluator.score_pair(claim, trace_a, trace_b)
    """

    def __init__(
        self,
        model_path: str,
        tokenizer_path: str,
        base_model: str,
        quantization_config=None,
        token: str = None,
        device: str = "cuda",
        max_length: int = 256,
    ):
        self.device     = torch.device(device if torch.cuda.is_available() else "cpu")
        self.max_length = max_length

        self.tokenizer = AutoTokenizer.from_pretrained(tokenizer_path, token=token)
        self.tokenizer.pad_token = self.tokenizer.eos_token

        self.model = CustomClassifier(
            base_model,
            num_labels=1,
            use_lora=True,
            is_base_encoder=False,
            quantization_config=quantization_config,
            token=token,
        )
        self.model.load_state_dict(
            torch.load(model_path, map_location=self.device), strict=False
        )
        self.model.to(self.device)
        self.model.eval()

    def _encode(self, claim: str, trace: str):
        text = f"Claim: {claim}\nCandidate A: {trace}"
        enc  = self.tokenizer(
            text,
            truncation=True,
            padding="max_length",
            max_length=self.max_length,
            return_tensors="pt",
        )
        return (
            enc["input_ids"].to(self.device),
            enc["attention_mask"].to(self.device),
        )

    def reward(self, claim: str, trace: str) -> float:
        """Scalar BT reward for a single trace."""
        ids, mask = self._encode(claim, trace)
        with torch.no_grad():
            return float(self.model(ids, mask).item())

    def score_pair(self, claim: str, trace_a: str, trace_b: str) -> float:
        """
        BT pairwise score. Drop-in for PairwiseVerifierEvaluator.score_pair().
        Positive = trace_a preferred.
        """
        return self.reward(claim, trace_a) - self.reward(claim, trace_b)

    def score_all(self, claim: str, traces: list) -> list:
        """
        Compute BT reward for every trace in one pass.
        Returns a list of floats usable directly as score_list.
        This is O(N) forward passes — the most efficient inference mode.
        """
        rewards = []
        for trace in traces:
            rewards.append(self.reward(claim, trace))
        return rewards



In [ ]:

# ─────────────────────────────────────────────────────────────────────────────
#  Inference loop using BTVerifierEvaluator.score_all()
#  O(N) forward passes per claim — no round-robin or tournament needed.
#  The BT rewards ARE the score_list.
#  Paste this as an alternative to Cell 24 for the pure-BT ablation.
# ─────────────────────────────────────────────────────────────────────────────

bt_evaluator = BTVerifierEvaluator(
    model_path=f"/content/drive/MyDrive/CheckThat Task2/Spanish/model_bt",
    tokenizer_path=BASE_MODEL,
    base_model=BASE_MODEL,
    token=HF_ACCESS_TOKEN,
)

with open(VAL_CSV, "r") as f:
    test_data = json.load(f)

predictions_bt = []

for idx, sample in enumerate(tqdm(test_data, desc="BT scoring")):
    claim    = sample["claim"]
    verdicts = sample["Verdict_list"]
    traces   = [remove_label_pattern(t).split("Label:")[0]
                for t in sample["Reasoning_traces"]]

    # O(N) — one forward pass per trace, no comparisons needed
    score_list = bt_evaluator.score_all(claim, traces)

    best_idx     = int(np.argmax(score_list))
    best_verdict = verdicts[best_idx]

    predictions_bt.append({
        "query_id":         idx,
        "Claim":            claim,
        # "Label":            sample["label"],
        "Verdict_BoN":      best_verdict,
        "BoN_Verdict_list": verdicts,
        "Reasoning_traces": traces,
        "score_list":       score_list,
    })

output_path = (
    f"/content/drive/MyDrive/CheckThat Task2/{language}/"
    f"clef_predictions_bt_3b.json"
)
with open(output_path, "w") as fp:
    json.dump(predictions_bt, fp, indent=4)

print(f"BT predictions saved → {output_path}")
INPUT_PATH = output_path   # point evaluation cell here



Loading weights:   0%|          | 0/254 [00:00<?, ?it/s]

BT scoring: 100%|██████████| 1164/1164 [22:09<00:00,  1.14s/it]

BT predictions saved → /content/drive/MyDrive/CheckThat Task2/Spanish/clef_predictions_bt_3b.json


In [ ]:

# ─────────────────────────────────────────────────────────────────────────────
#  Ablation guide
#  ─────────────────────────────────────────────────────────────────────────────
#
#  Three inference modes to compare after BT retraining:
#
#  A. Pure BT (above):
#       BTVerifierEvaluator.score_all()
#       O(N) calls, no tournament, BT rewards as score_list directly
#       → clef_predictions_bt.json
#
#  B. BT model + your existing PRP + tournament:
#       Load model_bt into PairwiseVerifierEvaluator (same interface)
#       Run tournament_inference.py with score_pair_prp()
#       → clef_predictions_bt_prp_tournament.json
#       Change: model_path=f"{OUTPUT_DIR}/model_bt" in PairwiseVerifierEvaluator
#
#  C. BT model + PrePAIR (EN and AR only, no PRP):
#       Load model_bt into PairwiseVerifierEvaluator
#       Run prepair_corrected.py (pairwise_evaluator = bt model)
#       The pointwise evaluator (Stage 1 seeding) stays as the original
#       pointwise model — only the comparison model is replaced
#       → clef_predictions_bt_prepair.json
#
#  Expected result ordering: C ≥ B ≥ A for EN and AR.
#  For ES: B ≥ A (PrePAIR doesn't help on ES as we found earlier).

# Evaluation

Evaluate RM prediction quality on the CLEF dataset.

Metrics computed:
  1. IR-style Recall@k and Precision@k (k=1..5):
       Recall@k    = (# relevant in top-k) / (# relevant in full 15-trace list)
       Precision@k = (# relevant in top-k) / k
     A trace is relevant if its verdict matches the ground truth label.
     Per-sample values saved to per_sample_ir.csv; means saved to result.csv.
  2. Per-class precision, recall, F1 (based on Verdict_BoN vs Label).
  3. Macro and weighted precision, recall, F1.

Output:
  output/RM_prediction/result.csv          — average / aggregate metrics
  output/RM_prediction/per_sample_ir.csv   — per-sample Recall@k and Precision@k

In [ ]:
import csv
import json
import os

from sklearn.metrics import classification_report

INPUT_PATH      = f"/content/drive/MyDrive/CheckThat Task2/English/clef_predictions_prepair_tournament_english.json"
OUTPUT_PATH     = f"/content/drive/MyDrive/CheckThat Task2/{language}/result_prepair_tournament_english_test.csv"
PER_SAMPLE_PATH = f"/content/drive/MyDrive/CheckThat Task2/{language}/ir_prepair_tournament_english_test.csv"
CLASSES         = ["false", "true", "conflicting"]
K_VALUES        = list(range(1, 6))


with open(INPUT_PATH, encoding="utf-8") as f:
    raw_data = json.load(f)

seen: set = set()
unique_data = []
for entry in raw_data:
    if entry["Claim"] not in seen:
        seen.add(entry["Claim"])
        unique_data.append(entry)

print(f"Total unique claims: {len(unique_data)}")


# ── 1. IR-style Recall@k and Precision@k ─────────────────────────────────────
def ir_metrics_at_k(entry: dict, k: int) -> tuple[float, float]:
    """
    Compute IR-style Recall@k and Precision@k for a single claim.

    Recall@k    = (# relevant in top-k) / (# relevant in full 15-trace list)
    Precision@k = (# relevant in top-k) / k

    A trace is relevant when its verdict matches the ground truth label.
    If no trace in the full list is relevant, Recall@k is defined as 0.
    """
    # label    = entry["Label"].lower()
    verdicts = [v.lower() for v in entry["BoN_Verdict_list"]]

    ranked = sorted(range(len(entry["score_list"])),
                    key=lambda i: entry["score_list"][i],
                    reverse=True)

    total_relevant     = sum(1 for v in verdicts if v == label)
    top_k_verdicts     = [verdicts[i] for i in ranked[:k]]
    retrieved_relevant = sum(1 for v in top_k_verdicts if v == label)

    recall_k    = retrieved_relevant / total_relevant if total_relevant > 0 else 0.0
    precision_k = retrieved_relevant / k

    return recall_k, precision_k


# Accumulate per-sample records and running sums for means
per_sample_records = []
sum_recall    = {k: 0.0 for k in K_VALUES}
sum_precision = {k: 0.0 for k in K_VALUES}

for entry in unique_data:
    record = {"query_id": entry["query_id"], "claim": entry["Claim"][:80]}
    for k in K_VALUES:
        r, p = ir_metrics_at_k(entry, k)
        record[f"recall@{k}"]    = round(r, 4)
        record[f"precision@{k}"] = round(p, 4)
        sum_recall[k]    += r
        sum_precision[k] += p
    per_sample_records.append(record)

n = len(unique_data)
mean_recall    = {k: sum_recall[k]    / n for k in K_VALUES}
mean_precision = {k: sum_precision[k] / n for k in K_VALUES}


# ── 2 & 3. Classification metrics on Verdict_BoN ─────────────────────────────
# y_true = [e["Label"].lower()       for e in unique_data]
y_pred = [e["Verdict_BoN"].lower() for e in unique_data]

report = classification_report(
    # y_true, y_pred,
    labels=CLASSES,
    output_dict=True,
    zero_division=0,
)


# ── Save aggregate metrics → result.csv ───────────────────────────────────────
os.makedirs(os.path.dirname(OUTPUT_PATH), exist_ok=True)

with open(OUTPUT_PATH, "w", newline="", encoding="utf-8") as f:
    w = csv.writer(f)

    # Section 1 — IR ranking metrics
    w.writerow(["=== IR Ranking Metrics (RM-ranked reasoning traces) ==="])
    w.writerow(["k", "Mean Recall@k", "Mean Precision@k"])
    for k in K_VALUES:
        w.writerow([k, round(mean_recall[k], 4), round(mean_precision[k], 4)])
    w.writerow([])

    # Section 2 — Per-class classification metrics
    w.writerow(["=== Per-Class Metrics (Verdict_BoN vs Ground Truth) ==="])
    w.writerow(["Class", "Precision", "Recall", "F1-Score", "Support"])
    for cls in CLASSES:
        r = report[cls]
        w.writerow([cls,
                    round(r["precision"], 4),
                    round(r["recall"],    4),
                    round(r["f1-score"],  4),
                    int(r["support"])])
    w.writerow([])

    # Section 3 — Macro & weighted aggregates
    w.writerow(["=== Aggregate Metrics ==="])
    w.writerow(["Average", "Precision", "Recall", "F1-Score"])
    for avg_type in ["macro avg", "weighted avg"]:
        r = report[avg_type]
        w.writerow([avg_type,
                    round(r["precision"], 4),
                    round(r["recall"],    4),
                    round(r["f1-score"],  4)])


# ── Save per-sample IR metrics → per_sample_ir.csv ───────────────────────────
per_sample_header = (
    ["query_id", "claim"]
    + [f"recall@{k}"    for k in K_VALUES]
    + [f"precision@{k}" for k in K_VALUES]
)

with open(PER_SAMPLE_PATH, "w", newline="", encoding="utf-8") as f:
    w = csv.DictWriter(f, fieldnames=per_sample_header)
    w.writeheader()
    w.writerows(per_sample_records)


# ── Print summary ─────────────────────────────────────────────────────────────
print("\n--- IR Ranking Metrics ---")
print(f"  {'k':>3}  {'Mean Recall@k':>14}  {'Mean Precision@k':>16}")
for k in K_VALUES:
    print(f"  {k:>3}  {mean_recall[k]:>14.4f}  {mean_precision[k]:>16.4f}")

print("\n--- Per-Class Metrics (Verdict_BoN) ---")
for cls in CLASSES:
    r = report[cls]
    print(f"  {cls:>12s}:  P={r['precision']:.4f}  R={r['recall']:.4f}  F1={r['f1-score']:.4f}  n={int(r['support'])}")

print("\n--- Aggregate Metrics ---")
for avg_type in ["macro avg", "weighted avg"]:
    r = report[avg_type]
    print(f"  {avg_type}:  P={r['precision']:.4f}  R={r['recall']:.4f}  F1={r['f1-score']:.4f}")

print(f"\nAggregate results  -> {OUTPUT_PATH}")
print(f"Per-sample results -> {PER_SAMPLE_PATH}")


Total unique claims: 1708


KeyError: 'Label'

eval

In [ ]:
import csv
import json
import os
from sklearn.metrics import classification_report

# ── Paths & Configuration ───────────────────────────────────────────────────
LANGUAGE = "English"  # Adjust dynamically or define explicitly

INPUT_PATH = "/content/drive/MyDrive/CheckThat Task2/English/clef_predictions_prepair_tournament_english.json"
GOLDEN_LABELS_PATH = "/content/drive/MyDrive/CheckThat Task2/English/clef_english_test_gold_labels_final.json"  # <--- NEW FILE PATH

OUTPUT_PATH = f"/content/drive/MyDrive/CheckThat Task2/{LANGUAGE}/result_prepair_tournament_english_test.csv"
PER_SAMPLE_PATH = f"/content/drive/MyDrive/CheckThat Task2/{LANGUAGE}/ir_prepair_tournament_english_test.csv"

CLASSES = ["false", "true", "conflicting"]
K_VALUES = list(range(1, 6))

# ── 1. Load Files ────────────────────────────────────────────────────────────
# Load the model predictions file
with open(INPUT_PATH, encoding="utf-8") as f:
    raw_data = json.load(f)

# Load the separate golden labels file
with open(GOLDEN_LABELS_PATH, encoding="utf-8") as f:
    raw_golden_data = json.load(f)

# ── 2. Create Reference Dictionary for Golden Labels ───────────────────────
# We use 'query_id' as a reliable mapping key.
# We also clean and normalize the labels to match your CLASSES definitions.
gold_lookup = {}
for entry in raw_golden_data:
    q_id = entry["query_id"]
    raw_label = entry["Label"].lower()

    # Optional alignment layer: If your golden dataset uses "supports"/"refutes"
    # instead of "true"/"false", uncomment and adjust this mapping:
    # if raw_label == "supports": raw_label = "true"
    # if raw_label == "refutes": raw_label = "false"

    gold_lookup[q_id] = raw_label

# ── 3. Filter Unique Claims From Predictions ─────────────────────────────────
seen = set()
unique_data = []
for entry in raw_data:
    if entry["Claim"] not in seen:
        seen.add(entry["Claim"])
        unique_data.append(entry)

print(f"Total unique claims: {len(unique_data)}")


# ── 4. IR Metrics Calculator Function ────────────────────────────────────────
def ir_metrics_at_k(entry: dict, k: int, label: str) -> tuple[float, float]:
    """
    Compute IR-style Recall@k and Precision@k for a single claim using its
    externally supplied golden label.
    """
    verdicts = [v.lower() for v in entry["BoN_Verdict_list"]]

    ranked = sorted(
        range(len(entry["score_list"])),
        key=lambda i: entry["score_list"][i],
        reverse=True
    )

    total_relevant = sum(1 for v in verdicts if v == label)
    top_k_verdicts = [verdicts[i] for i in ranked[:k]]
    retrieved_relevant = sum(1 for v in top_k_verdicts if v == label)

    recall_k = retrieved_relevant / total_relevant if total_relevant > 0 else 0.0
    precision_k = retrieved_relevant / k

    return recall_k, precision_k


# ── 5. Run Evaluations ───────────────────────────────────────────────────────
per_sample_records = []
sum_recall = {k: 0.0 for k in K_VALUES}
sum_precision = {k: 0.0 for k in K_VALUES}

y_true = []
y_pred = []

for entry in unique_data:
    q_id = entry["query_id"]

    # Retrieve the external golden label or skip/default if it doesn't match
    if q_id not in gold_lookup:
        print(f"Warning: query_id {q_id} missing from Golden Labels file. Skipping.")
        continue

    label = gold_lookup[q_id]

    # Build IR sample records
    record = {"query_id": q_id, "claim": entry["Claim"][:80]}
    for k in K_VALUES:
        r, p = ir_metrics_at_k(entry, k, label)
        record[f"recall@{k}"] = round(r, 4)
        record[f"precision@{k}"] = round(p, 4)
        sum_recall[k] += r
        sum_precision[k] += p
    per_sample_records.append(record)

    # Collect lists for classification metrics
    y_true.append(label)
    y_pred.append(entry["Verdict_BoN"].lower())

# Recalculate 'n' based on explicitly matched entries
n = len(per_sample_records)
mean_recall = {k: sum_recall[k] / n for k in K_VALUES}
mean_precision = {k: sum_precision[k] / n for k in K_VALUES}

# Generate classification report
report = classification_report(
    y_true,
    y_pred,
    labels=CLASSES,
    output_dict=True,
    zero_division=0,
)


# ── 6. Save Aggregate Metrics → result.csv ───────────────────────────────────
os.makedirs(os.path.dirname(OUTPUT_PATH), exist_ok=True)

with open(OUTPUT_PATH, "w", newline="", encoding="utf-8") as f:
    w = csv.writer(f)

    # Section 1 — IR ranking metrics
    w.writerow(["=== IR Ranking Metrics (RM-ranked reasoning traces) ==="])
    w.writerow(["k", "Mean Recall@k", "Mean Precision@k"])
    for k in K_VALUES:
        w.writerow([k, round(mean_recall[k], 4), round(mean_precision[k], 4)])
    w.writerow([])

    # Section 2 — Per-class classification metrics
    w.writerow(["=== Per-Class Metrics (Verdict_BoN vs Ground Truth) ==="])
    w.writerow(["Class", "Precision", "Recall", "F1-Score", "Support"])
    for cls in CLASSES:
        r = report[cls]
        w.writerow([
            cls,
            round(r["precision"], 4),
            round(r["recall"], 4),
            round(r["f1-score"], 4),
            int(r["support"])
        ])
    w.writerow([])

    # Section 3 — Macro & weighted aggregates
    w.writerow(["=== Aggregate Metrics ==="])
    w.writerow(["Average", "Precision", "Recall", "F1-Score"])
    for avg_type in ["macro avg", "weighted avg"]:
        r = report[avg_type]
        w.writerow([
            avg_type,
            round(r["precision"], 4),
            round(r["recall"], 4),
            round(r["f1-score"], 4)
        ])


# ── 7. Save per-sample IR metrics → per_sample_ir.csv ───────────────────────
per_sample_header = (
    ["query_id", "claim"]
    + [f"recall@{k}" for k in K_VALUES]
    + [f"precision@{k}" for k in K_VALUES]
)

with open(PER_SAMPLE_PATH, "w", newline="", encoding="utf-8") as f:
    w = csv.DictWriter(f, fieldnames=per_sample_header)
    w.writeheader()
    w.writerows(per_sample_records)


# ── 8. Print Console Summary ─────────────────────────────────────────────────
print("\n--- IR Ranking Metrics ---")
print(f"  {'k':>3}  {'Mean Recall@k':>14}  {'Mean Precision@k':>16}")
for k in K_VALUES:
    print(f"  {k:>3}  {mean_recall[k]:>14.4f}  {mean_precision[k]:>16.4f}")

print("\n--- Per-Class Metrics (Verdict_BoN) ---")
for cls in CLASSES:
    r = report[cls]
    print(f"  {cls:>12s}:  P={r['precision']:.4f}  R={r['recall']:.4f}  F1={r['f1-score']:.4f}  n={int(r['support'])}")

print("\n--- Aggregate Metrics ---")
for avg_type in ["macro avg", "weighted avg"]:
    r = report[avg_type]
    print(f"  {avg_type}:  P={r['precision']:.4f}  R={r['recall']:.4f}  F1={r['f1-score']:.4f}")

print(f"\nAggregate results  -> {OUTPUT_PATH}")
print(f"Per-sample results -> {PER_SAMPLE_PATH}")

Total unique claims: 1708

--- IR Ranking Metrics ---
    k   Mean Recall@k  Mean Precision@k
    1          0.0496            0.6493
    2          0.0880            0.6341
    3          0.1246            0.6312
    4          0.1555            0.6099
    5          0.1913            0.6119

--- Per-Class Metrics (Verdict_BoN) ---
         false:  P=0.7736  R=0.8023  F1=0.7877  n=1244
          true:  P=0.2645  R=0.2580  F1=0.2612  n=283
   conflicting:  P=0.1481  R=0.1088  F1=0.1255  n=147

--- Aggregate Metrics ---
  macro avg:  P=0.3954  R=0.3897  F1=0.3915
  weighted avg:  P=0.6326  R=0.6493  F1=0.6405

Aggregate results  -> /content/drive/MyDrive/CheckThat Task2/English/result_prepair_tournament_english_test.csv
Per-sample results -> /content/drive/MyDrive/CheckThat Task2/English/ir_prepair_tournament_english_test.csv


In [ ]:
import json

INPUT_PATH = "/content/drive/MyDrive/CheckThat Task2/English/clef_predictions_prepair_tournament_english.json"
GOLDEN_LABELS_PATH = "/content/drive/MyDrive/CheckThat Task2/English/clef_english_test_gold_labels_final.json"

# 1. Load data
with open(INPUT_PATH, encoding="utf-8") as f:
    predictions = json.load(f)

with open(GOLDEN_LABELS_PATH, encoding="utf-8") as f:
    golden = json.load(f)

# 2. Extract IDs and Types
pred_ids = {entry.get("query_id") for entry in predictions if "query_id" in entry}
gold_ids = {entry.get("query_id") for entry in golden if "query_id" in entry}

pred_id_types = {type(qid) for qid in pred_ids}
gold_id_types = {type(qid) for qid in gold_ids}

print("=== Type Diagnostics ===")
print(f"Prediction query_id data types found: {pred_id_types}")
print(f"Golden Labels query_id data types found: {gold_id_types}")

# 3. String Conversion Check (Simulate fix for Type Mismatch)
normalized_pred_ids = {str(qid).strip() for qid in pred_ids}
normalized_gold_ids = {str(qid).strip() for qid in gold_ids}
missing_after_string_cast = normalized_pred_ids - normalized_gold_ids

print(f"\nTotal IDs in predictions: {len(pred_ids)}")
print(f"Total IDs in golden labels: {len(gold_ids)}")

# 4. Analyze specific missing items from your log
sample_missing = [115, 150, 155, 163, 165]
print(f"\n=== Deep Dive Into Selected Missing IDs ===")

for target_id in sample_missing:
    print(f"\n--- Investigating ID: {target_id} ---")

    # Find matching entries in prediction file
    pred_matches = [e for e in predictions if str(e.get("query_id")) == str(target_id)]
    if pred_matches:
        p_entry = pred_matches[0]
        p_claim = p_entry.get("Claim", "").strip()
        print(f"  [Prediction File] Found! Claim snapshot: '{p_claim[:60]}...'")

        # Check if the golden file has this ID under ANY type
        gold_match_by_id = [e for e in golden if str(e.get("query_id")) == str(target_id)]
        if gold_match_by_id:
            print(f"  [Golden File] ID exists, but type mismatch occurred! Golden type is {type(gold_match_by_id[0].get('query_id'))}")
        else:
            print(f"  [Golden File] ID does NOT exist explicitly.")

            # Check if the claim text itself exists in the golden file under a DIFFERENT query_id
            gold_match_by_claim = [e for e in golden if e.get("claim", "").strip().lower() == p_claim.lower()]
            if gold_match_by_claim:
                g_entry = gold_match_by_claim[0]
                print(f"  [Match Found By Text!] The text exists in Golden file but uses query_id: {g_entry.get('query_id')}")
            else:
                print(f"  [Match Failed] This claim text is also entirely missing from the Golden file.")
    else:
        print(f"  ID {target_id} not even found in predictions raw loop.")

print("\n==============================")
if len(missing_after_string_cast) == 0 and (len(pred_id_types) > 1 or len(gold_id_types) > 1 or pred_id_types != gold_id_types):
    print("💡 DIAGNOSIS: You have a clean data-type mismatch (ints vs strings). Casting both fields to string during lookup will resolve all warnings.")
else:
    print(f"💡 DIAGNOSIS: {len(missing_after_string_cast)} keys are structurally completely absent from your golden file matching targets.")

=== Type Diagnostics ===
Prediction query_id data types found: {<class 'int'>}
Golden Labels query_id data types found: {<class 'int'>}

Total IDs in predictions: 2558
Total IDs in golden labels: 2558

=== Deep Dive Into Selected Missing IDs ===

--- Investigating ID: 115 ---
  [Prediction File] Found! Claim snapshot: 'Video of vote rigging in Uganda's 2026 elections...'
  [Golden File] ID does NOT exist explicitly.
  [Match Found By Text!] The text exists in Golden file but uses query_id: 116

--- Investigating ID: 150 ---
  [Prediction File] Found! Claim snapshot: '"Public facilities account now for over 45% of all health fa...'
  [Golden File] ID does NOT exist explicitly.
  [Match Found By Text!] The text exists in Golden file but uses query_id: 152

--- Investigating ID: 155 ---
  [Prediction File] Found! Claim snapshot: '"I am aware that I will be taking over a nation where 1.5 ba...'
  [Golden File] ID does NOT exist explicitly.
  [Match Found By Text!] The text exists in Golden

In [ ]:
import csv
import json
import os
from sklearn.metrics import classification_report

# ── Paths & Configuration ───────────────────────────────────────────────────
LANGUAGE = "English"

INPUT_PATH = "/content/drive/MyDrive/CheckThat Task2/English/clef_predictions_prepair_tournament_english.json"
GOLDEN_LABELS_PATH = "/content/drive/MyDrive/CheckThat Task2/English/clef_english_test_gold_labels_final.json"

OUTPUT_PATH = f"/content/drive/MyDrive/CheckThat Task2/{LANGUAGE}/result_prepair_tournament_english_test.csv"
PER_SAMPLE_PATH = f"/content/drive/MyDrive/CheckThat Task2/{LANGUAGE}/ir_prepair_tournament_english_test.csv"

CLASSES = ["false", "true", "conflicting"]
K_VALUES = list(range(1, 6))

# ── 1. Load Files ────────────────────────────────────────────────────────────
with open(INPUT_PATH, encoding="utf-8") as f:
    raw_data = json.load(f)

with open(GOLDEN_LABELS_PATH, encoding="utf-8") as f:
    raw_golden_data = json.load(f)

# ── 2. Create Reference Dictionary via Normalized Text ───────────────────────
# We map directly using the raw text string to handle the ID discrepancies.
gold_lookup = {}
for entry in raw_golden_data:
    # Golden labels file uses lowercase 'claim'
    claim_text = entry.get("claim", "").strip().lower()
    if claim_text:
        raw_label = entry["Label"].lower()

        # Alignment mapping for text label variance
        if raw_label == "supports":
            raw_label = "true"
        elif raw_label == "refutes":
            raw_label = "false"

        gold_lookup[claim_text] = raw_label

# ── 3. Filter Unique Claims From Predictions ─────────────────────────────────
seen = set()
unique_data = []
for entry in raw_data:
    # Predictions file uses capitalized 'Claim'
    norm_claim = entry.get("Claim", "").strip().lower()
    if norm_claim not in seen:
        seen.add(norm_claim)
        unique_data.append(entry)

print(f"Total unique claims from prediction file: {len(unique_data)}")


# ── 4. IR Metrics Calculator Function ────────────────────────────────────────
def ir_metrics_at_k(entry: dict, k: int, label: str) -> tuple[float, float]:
    verdicts = [v.lower() for v in entry["BoN_Verdict_list"]]

    ranked = sorted(
        range(len(entry["score_list"])),
        key=lambda i: entry["score_list"][i],
        reverse=True
    )

    total_relevant = sum(1 for v in verdicts if v == label)
    top_k_verdicts = [verdicts[i] for i in ranked[:k]]
    retrieved_relevant = sum(1 for v in top_k_verdicts if v == label)

    recall_k = retrieved_relevant / total_relevant if total_relevant > 0 else 0.0
    precision_k = retrieved_relevant / k

    return recall_k, precision_k


# ── 5. Run Evaluations ───────────────────────────────────────────────────────
per_sample_records = []
sum_recall = {k: 0.0 for k in K_VALUES}
sum_precision = {k: 0.0 for k in K_VALUES}

y_true = []
y_pred = []
missing_count = 0

for entry in unique_data:
    norm_claim = entry.get("Claim", "").strip().lower()

    # Match using normalized text string instead of id
    if norm_claim not in gold_lookup:
        missing_count += 1
        continue

    label = gold_lookup[norm_claim]

    # Build IR sample records
    record = {"query_id": entry["query_id"], "claim": entry["Claim"][:80]}
    for k in K_VALUES:
        r, p = ir_metrics_at_k(entry, k, label)
        record[f"recall@{k}"] = round(r, 4)
        record[f"precision@{k}"] = round(p, 4)
        sum_recall[k] += r
        sum_precision[k] += p
    per_sample_records.append(record)

    # Collect lists for classification metrics
    y_true.append(label)
    y_pred.append(entry["Verdict_BoN"].lower())

if missing_count > 0:
    print(f"⚠️ Successfully aligned dataset. {missing_count} claims were completely missing from the golden set text definitions and omitted.")

# Recalculate metrics array length based on successfully matched records
n = len(per_sample_records)
mean_recall = {k: sum_recall[k] / n for k in K_VALUES}
mean_precision = {k: sum_precision[k] / n for k in K_VALUES}

# Generate classification report
report = classification_report(
    y_true,
    y_pred,
    labels=CLASSES,
    output_dict=True,
    zero_division=0,
)


# ── 6. Save Aggregate Metrics → result.csv ───────────────────────────────────
os.makedirs(os.path.dirname(OUTPUT_PATH), exist_ok=True)

with open(OUTPUT_PATH, "w", newline="", encoding="utf-8") as f:
    w = csv.writer(f)

    w.writerow(["=== IR Ranking Metrics (RM-ranked reasoning traces) ==="])
    w.writerow(["k", "Mean Recall@k", "Mean Precision@k"])
    for k in K_VALUES:
        w.writerow([k, round(mean_recall[k], 4), round(mean_precision[k], 4)])
    w.writerow([])

    w.writerow(["=== Per-Class Metrics (Verdict_BoN vs Ground Truth) ==="])
    w.writerow(["Class", "Precision", "Recall", "F1-Score", "Support"])
    for cls in CLASSES:
        r = report[cls]
        w.writerow([
            cls,
            round(r["precision"], 4),
            round(r["recall"], 4),
            round(r["f1-score"], 4),
            int(r["support"])
        ])
    w.writerow([])

    w.writerow(["=== Aggregate Metrics ==="])
    w.writerow(["Average", "Precision", "Recall", "F1-Score"])
    for avg_type in ["macro avg", "weighted avg"]:
        r = report[avg_type]
        w.writerow([
            avg_type,
            round(r["precision"], 4),
            round(r["recall"], 4),
            round(r["f1-score"], 4)
        ])


# ── 7. Save per-sample IR metrics → per_sample_ir.csv ───────────────────────
per_sample_header = (
    ["query_id", "claim"]
    + [f"recall@{k}" for k in K_VALUES]
    + [f"precision@{k}" for k in K_VALUES]
)

with open(PER_SAMPLE_PATH, "w", newline="", encoding="utf-8") as f:
    w = csv.DictWriter(f, fieldnames=per_sample_header)
    w.writeheader()
    w.writerows(per_sample_records)


# ── 8. Print Console Summary ─────────────────────────────────────────────────
print("\n--- IR Ranking Metrics ---")
print(f"  {'k':>3}  {'Mean Recall@k':>14}  {'Mean Precision@k':>16}")
for k in K_VALUES:
    print(f"  {k:>3}  {mean_recall[k]:>14.4f}  {mean_precision[k]:>16.4f}")

print("\n--- Per-Class Metrics (Verdict_BoN) ---")
for cls in CLASSES:
    r = report[cls]
    print(f"  {cls:>12s}:  P={r['precision']:.4f}  R={r['recall']:.4f}  F1={r['f1-score']:.4f}  n={int(r['support'])}")

print("\n--- Aggregate Metrics ---")
for avg_type in ["macro avg", "weighted avg"]:
    r = report[avg_type]
    print(f"  {avg_type}:  P={r['precision']:.4f}  R={r['recall']:.4f}  F1={r['f1-score']:.4f}")

print(f"\nAggregate results  -> {OUTPUT_PATH}")
print(f"Per-sample results -> {PER_SAMPLE_PATH}")

Total unique claims from prediction file: 1708

--- IR Ranking Metrics ---
    k   Mean Recall@k  Mean Precision@k
    1          0.0556            0.8074
    2          0.1034            0.7948
    3          0.1481            0.7931
    4          0.1878            0.7709
    5          0.2320            0.7730

--- Per-Class Metrics (Verdict_BoN) ---
         false:  P=0.8823  R=0.9114  F1=0.8966  n=1275
          true:  P=0.6857  R=0.6906  F1=0.6882  n=278
   conflicting:  P=0.2252  R=0.1613  F1=0.1880  n=155

--- Aggregate Metrics ---
  macro avg:  P=0.5977  R=0.5878  F1=0.5909
  weighted avg:  P=0.7907  R=0.8074  F1=0.7984

Aggregate results  -> /content/drive/MyDrive/CheckThat Task2/English/result_prepair_tournament_english_test.csv
Per-sample results -> /content/drive/MyDrive/CheckThat Task2/English/ir_prepair_tournament_english_test.csv
